# 01_07_build_amortization_drp

Тетрадка для полного пересчета амортизации по терминалам за период Jan-Jul 2026
(граница периода: `2026-08-01`) и полной перезаписи таблицы
`sandbox_ai.shestopalov_terminal_amortization_model`.

Пайплайн:
1. Выгрузка активных терминалов по месяцам периода из Lake.
2. Выгрузка моделей терминалов из CDWH (`BMRT.BM_DET_DEVICE`) батчами.
3. Join с ценами моделей из `term_model.xlsx`.
4. Расчет `amortization_monthly = price / 48`, окна 48 месяцев и `amortization_for_report_month`.
5. Полная перезагрузка целевой таблицы в Datalake/Impala (`DROP + CREATE + ORC load`).

6. **Исключение моделей из начисления** (`exclude_amort_by_model`):
   бренды Tactilion / VeriFone / Ingenico / IRAS (все модели);
   Pax — только s80 / s90 / D210. Для них `amortization_for_report_month = 0`.
   Окно с даты выдачи для остальных моделей без изменений.


In [ ]:
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Параметры периода: Jan-Jul 2026 (граница периода: 2026-08-01)
period_start = '2026-01-01'
period_end_exclusive = '2026-08-01'
period_months = pd.date_range(period_start, pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1), freq='MS')

term_model_excel_path = '/home/jovyan/documents/Equaring/Data/term_model.xlsx'
source_file = 'term_model.xlsx'

target_table = 'sandbox_ai.shestopalov_terminal_amortization_model'
save_table_orc_name = 'shestopalov_terminal_amortization_model_2026_01_2026_06.orc'

# CDWH credentials (as agreed earlier)
khd_user = 'DS_LII'
khd_password = 'dl3S$wolx5dz'

print('period_months =', [m.strftime('%Y-%m') for m in period_months])
print('term_model_excel_path =', term_model_excel_path)
print('target_table =', target_table)


def clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in {'None', 'nan', 'NaN'}:
            out.append(s)
    return sorted(set(out))


def sql_in(values):
    vals = clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def norm_model(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().lower().replace('\xa0', ' ')
    if s in {'none', 'nan', 'null', '<null>', 'nat'}:
        return None
    s = s.replace('ё', 'е')
    s = re.sub(r'\s+', ' ', s)
    return s if s else None


def norm_terminal_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s.upper()




def clean_cdwh_text(v):
    """Normalize CDWH text: real nulls stay NaN (never the string 'None'/'nan')."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() in {"none", "nan", "null", "<null>", "nat"}:
        return np.nan
    return s


def is_blank_series(s):
    """True where value is null / empty / literal None|nan|null."""
    def _blank(v):
        if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
            return True
        t = str(v).strip()
        return t == "" or t.lower() in {"none", "nan", "null", "<null>", "nat"}

    return s.map(_blank)

def iter_chunks(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def split_nter_for_id_and_code(values):
    id_numeric = []
    code_values = []
    for v in values:
        s = str(v).strip()
        if not s:
            continue
        code_values.append(s)
        if re.fullmatch(r'\d+', s):
            id_numeric.append(str(int(s)))
    return sorted(set(id_numeric)), sorted(set(code_values))


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

dl = connect(
    to='DATALAKE',
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
dl._init_connection()

cdwh_connection = connect(
    to='CDWH',
    user_params={
        'user_name': khd_user,
        'password': khd_password,
    }
)
cdwh_connection._init_connection()

print('Impala + Datalake + CDWH initialized')

In [ ]:
# 1) Активные терминалы по месяцам периода + first_d_ter_delivery по serial
month_rows_sql = []
for m in period_months:
    month_start = m.strftime('%Y-%m-%d')
    month_end = (m + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    month_rows_sql.append(
        f"select cast('{month_start}' as date) as snapshot_month_start, cast('{month_end}' as date) as snapshot_month_end"
    )
months_sql = '\nunion all\n'.join(month_rows_sql)

sql_terminal_months = f"""
with months as (
{months_sql}
),
base as (
    select
      cast(t.c_nter as string) as c_nter,
      cast(t.c_pos_serial as string) as c_pos_serial,
      cast(t.d_ter_install as date) as d_ter_install,
      cast(t.d_ter_close as date) as d_ter_close,
      cast(t.d_ter_delivery as date) as d_ter_delivery
    from ods_alpha.scd1_pos_terminals t
    where t.c_nter is not null
      and t.c_pos_serial is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
      and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) < cast('{period_end_exclusive}' as date)
      and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{period_start}' as date)
),
first_deliver as (
    select
      cast(t.c_pos_serial as string) as c_pos_serial,
      min(cast(t.d_ter_delivery as date)) as first_d_ter_delivery
    from ods_alpha.scd1_pos_terminals t
    where t.c_pos_serial is not null
      and t.d_ter_delivery is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
    group by cast(t.c_pos_serial as string)
),
active_raw as (
    select
      cast(m.snapshot_month_start as date) as snapshot_month_start,
      cast(b.c_nter as string) as c_nter,
      cast(b.c_pos_serial as string) as c_pos_serial,
      cast(fd.first_d_ter_delivery as date) as first_d_ter_delivery,
      cast(b.d_ter_install as date) as d_ter_install,
      cast(b.d_ter_close as date) as d_ter_close,
      row_number() over (
        partition by cast(m.snapshot_month_start as date), cast(b.c_nter as string)
        order by
          coalesce(cast(b.d_ter_install as date), cast('1900-01-01' as date)) desc,
          coalesce(cast(b.d_ter_close as date), cast('2999-12-31' as date)) desc,
          cast(b.c_pos_serial as string) desc
      ) as rn
    from months m
    join base b
      on coalesce(cast(b.d_ter_install as date), cast('1900-01-01' as date)) <= m.snapshot_month_end
     and coalesce(cast(b.d_ter_close as date), cast('2999-12-31' as date)) >= m.snapshot_month_start
    left join first_deliver fd
      on fd.c_pos_serial = b.c_pos_serial
)
select
  cast(snapshot_month_start as string) as snapshot_month_start,
  cast(c_nter as string) as c_nter,
  cast(c_pos_serial as string) as c_pos_serial,
  cast(first_d_ter_delivery as date) as first_d_ter_delivery,
  cast(d_ter_install as date) as d_ter_install,
  cast(d_ter_close as date) as d_ter_close
from active_raw
where rn = 1
"""

with imp:
    imp.execute('set MEM_LIMIT=16g')
    terminals_monthly_df = imp.fetch(sql_terminal_months)

if terminals_monthly_df is None:
    terminals_monthly_df = pd.DataFrame(
        columns=['snapshot_month_start', 'c_nter', 'c_pos_serial', 'first_d_ter_delivery', 'd_ter_install', 'd_ter_close']
    )

for c in ['snapshot_month_start', 'c_nter', 'c_pos_serial']:
    if c in terminals_monthly_df.columns:
        terminals_monthly_df[c] = terminals_monthly_df[c].astype(str).str.strip()

terminals_monthly_df['nter_norm'] = terminals_monthly_df['c_nter'].apply(norm_terminal_key)

print('terminals_monthly rows =', len(terminals_monthly_df))
print('distinct c_nter =', terminals_monthly_df['c_nter'].nunique() if len(terminals_monthly_df) else 0)
display(terminals_monthly_df.head(10))

In [ ]:
# 2) Модели терминалов из CDWH (Oracle-safe batching)
nter_values = clean_keys(terminals_monthly_df['c_nter'].tolist()) if len(terminals_monthly_df) else []
id_values_numeric, code_values = split_nter_for_id_and_code(nter_values)

cdwh_id_batches = 0
cdwh_code_batches = 0
cdwh_code_fallback_batches = 0
bm_chunks = []

if nter_values:
    with cdwh_connection:
        if id_values_numeric:
            for id_chunk in iter_chunks(id_values_numeric, 950):
                id_in = ', '.join(id_chunk)
                sql_by_id = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.ID_DEVICE in ({id_in})
                """
                part_df = cdwh_connection.fetch(sql_by_id)
                cdwh_id_batches += 1
                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

        if code_values:
            for code_chunk in iter_chunks(code_values, 900):
                code_in = sql_in(code_chunk)
                sql_by_code_fast = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.CODE_DEVICE in ({code_in})
                """
                try:
                    part_df = cdwh_connection.fetch(sql_by_code_fast)
                    cdwh_code_batches += 1
                except Exception as exc:
                    sql_by_code_fallback = f"""
                    select distinct
                      trim(to_char(d.ID_DEVICE)) as device,
                      trim(to_char(d.CODE_DEVICE)) as code_device,
                      trim(to_char(d.MODEL_DEVICE)) as model_device
                    from BMRT.BM_DET_DEVICE d
                    where trim(to_char(d.CODE_DEVICE)) in ({code_in})
                    """
                    part_df = cdwh_connection.fetch(sql_by_code_fallback)
                    cdwh_code_fallback_batches += 1
                    print(f'CODE_DEVICE fallback chunk due to: {type(exc).__name__}')

                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

if bm_chunks:
    bm_det_device_df = pd.concat(bm_chunks, ignore_index=True)
else:
    bm_det_device_df = pd.DataFrame(columns=['device', 'code_device', 'model_device'])

bm_det_device_df.columns = [str(c).strip().lower() for c in bm_det_device_df.columns]
for col in ['device', 'code_device', 'model_device']:
    if col not in bm_det_device_df.columns:
        bm_det_device_df[col] = np.nan
    bm_det_device_df[col] = bm_det_device_df[col].map(clean_cdwh_text)

bm_det_device_df['device_norm'] = bm_det_device_df['device'].apply(norm_terminal_key)
bm_det_device_df['code_device_norm'] = bm_det_device_df['code_device'].apply(norm_terminal_key)
_before_bm = len(bm_det_device_df)
bm_det_device_df = bm_det_device_df.loc[bm_det_device_df['model_device'].notna()].copy()
print(f'bm_det_device: dropped empty MODEL_DEVICE rows: {_before_bm - len(bm_det_device_df):,}')
bm_det_device_df = bm_det_device_df.drop_duplicates(subset=['device', 'code_device', 'model_device'])

print('CDWH batch stats:')
print('  by_id =', cdwh_id_batches)
print('  by_code_fast =', cdwh_code_batches)
print('  by_code_fallback =', cdwh_code_fallback_batches)
print('bm_det_device rows =', len(bm_det_device_df))
display(bm_det_device_df.head(10))


In [ ]:
# 3) Цены моделей из Excel + join к terminal-month perimeter
price_src_df = pd.read_excel(term_model_excel_path)
required_price_cols = ['model_name', 'price']
missing_price_cols = [c for c in required_price_cols if c not in price_src_df.columns]
if missing_price_cols:
    raise RuntimeError(f'В term_model.xlsx отсутствуют колонки: {missing_price_cols}')

price_map_df = price_src_df[['model_name', 'price']].copy()
price_map_df['model_key'] = price_map_df['model_name'].apply(norm_model)
price_map_df['price'] = pd.to_numeric(price_map_df['price'], errors='coerce')
price_map_df = (
    price_map_df.dropna(subset=['model_key', 'price'])
    .groupby('model_key', as_index=False)
    .agg(price=('price', 'max'))
)

# Model mapping priority: ID_DEVICE first, then CODE_DEVICE
if len(bm_det_device_df):
    device_map_df = (
        bm_det_device_df.loc[
            bm_det_device_df['device_norm'].notna()
            & bm_det_device_df['model_device'].notna(),
            ['device_norm', 'model_device']
        ]
        .drop_duplicates(subset=['device_norm'], keep='first')
    )
    code_map_df = (
        bm_det_device_df.loc[
            bm_det_device_df['code_device_norm'].notna()
            & bm_det_device_df['model_device'].notna(),
            ['code_device_norm', 'model_device']
        ]
        .drop_duplicates(subset=['code_device_norm'], keep='first')
    )
    device_map = dict(zip(device_map_df['device_norm'], device_map_df['model_device']))
    code_map = dict(zip(code_map_df['code_device_norm'], code_map_df['model_device']))
else:
    device_map = {}
    code_map = {}

amort_base_df = terminals_monthly_df.copy()
amort_base_df['model_device'] = amort_base_df['nter_norm'].map(device_map)
amort_base_df['join_key_used'] = np.where(
    amort_base_df['model_device'].notna(), 'id_device', None
)

need_code_mask = amort_base_df['model_device'].isna()
amort_base_df.loc[need_code_mask, 'model_device'] = amort_base_df.loc[need_code_mask, 'nter_norm'].map(code_map)
amort_base_df.loc[need_code_mask & amort_base_df['model_device'].notna(), 'join_key_used'] = 'code_device'

# sanitize again (never keep literal 'None' / 'nan')
amort_base_df['model_device'] = amort_base_df['model_device'].map(clean_cdwh_text)
amort_base_df.loc[amort_base_df['model_device'].isna(), 'join_key_used'] = None

amort_base_df['model_key'] = amort_base_df['model_device'].apply(norm_model)
amort_base_df.loc[amort_base_df['model_key'].isna(), ['model_device', 'join_key_used']] = np.nan

amort_base_df = amort_base_df.merge(price_map_df, on='model_key', how='left')

print('price_map rows =', len(price_map_df))
print('amort_base rows =', len(amort_base_df))
print(
    'model found =', int(amort_base_df['model_device'].notna().sum()),
    '| price found =', int(pd.to_numeric(amort_base_df['price'], errors='coerce').notna().sum()),
)
display(amort_base_df.head(10))


In [ ]:
# 4) Расчет амортизации по месяцам
amort_df = amort_base_df.copy()

amort_df['snapshot_month_start'] = pd.to_datetime(amort_df['snapshot_month_start'], errors='coerce')
amort_df['first_d_ter_delivery'] = pd.to_datetime(amort_df['first_d_ter_delivery'], errors='coerce')

amort_df['missing_serial'] = is_blank_series(amort_df['c_pos_serial'])
amort_df['missing_deliver'] = amort_df['first_d_ter_delivery'].isna()
# missing_model: no usable CDWH model (null / literal None / bad model_key)
amort_df['missing_model'] = (
    is_blank_series(amort_df['model_device'])
    | amort_df['model_key'].isna()
)
# missing_price: model known, but no Excel price (true CDWH→Excel gap)
amort_df['missing_price'] = (~amort_df['missing_model']) & pd.to_numeric(amort_df['price'], errors='coerce').isna()


def qc_status_fn(row):
    if row['missing_serial']:
        return 'missing_serial'
    if row['missing_deliver']:
        return 'missing_deliver'
    if row['missing_model']:
        return 'missing_model'
    if row['missing_price']:
        return 'missing_price'
    return 'complete'


amort_df['qc_status'] = amort_df.apply(qc_status_fn, axis=1)

complete_mask = amort_df['qc_status'] == 'complete'

first_month_first_day = amort_df.loc[complete_mask, 'first_d_ter_delivery'].dt.to_period('M').dt.to_timestamp()
report_month_first_day = amort_df.loc[complete_mask, 'snapshot_month_start'].dt.to_period('M').dt.to_timestamp()

months_from_start = (
    (report_month_first_day.dt.year - first_month_first_day.dt.year) * 12
    + (report_month_first_day.dt.month - first_month_first_day.dt.month)
)

amort_df['months_from_start'] = np.nan
amort_df.loc[complete_mask, 'months_from_start'] = months_from_start

amort_df['is_in_48m_window'] = (
    amort_df['months_from_start'].notna()
    & (amort_df['months_from_start'] >= 0)
    & (amort_df['months_from_start'] < 48)
)

amort_df['amortization_monthly'] = pd.to_numeric(amort_df['price'], errors='coerce') / 48.0
amort_df['amortization_for_report_month'] = amort_df['amortization_monthly'].where(amort_df['is_in_48m_window'], 0.0)
amort_df['amortization_for_report_month'] = amort_df['amortization_for_report_month'].fillna(0.0)

amort_df['report_month'] = amort_df['snapshot_month_start'].dt.strftime('%Y-%m')
amort_df['snapshot_month_start'] = amort_df['snapshot_month_start'].dt.strftime('%Y-%m-%d')
amort_df['first_d_ter_delivery'] = amort_df['first_d_ter_delivery'].dt.strftime('%Y-%m-%d')

amort_df['is_in_48m_window'] = amort_df['is_in_48m_window'].astype(int)
amort_df['is_amortized'] = (amort_df['amortization_for_report_month'] > 0).astype(int)

# --- Exclude brands/models from amortization (business rule) ---
# Zero amort for brands: Tactilion, VeriFone, Ingenico, IRAS (all models).
# Zero amort for Pax models: s80, s90, D210 (name variants).
# Delivery-date / 48m window logic for other models is unchanged.

import re as _re_excl

_EXCLUDE_BRAND_RE = _re_excl.compile(
    r'(tactilion|verifone|veri\s*fone|ingenico|iras)',
    _re_excl.IGNORECASE,
)
# Pax family restricted to s80 / s90 / d210 (allow separators/suffixes)
_EXCLUDE_PAX_RE = _re_excl.compile(
    r'(?:^|[^a-z0-9])pax(?:[^a-z0-9]+|\s+).*(?:s\s*80|s\s*90|d\s*210)'
    r'|(?:^|[^a-z0-9])(?:s\s*80|s\s*90|d\s*210).*(?:^|[^a-z0-9])pax'
    r'|(?:^|[^a-z0-9])pax\s*(?:s\s*80|s\s*90|d\s*210)',
    _re_excl.IGNORECASE,
)


def _exclude_amort_by_model(model_device, model_key=None):
    text = ' '.join(
        str(x) for x in (model_device, model_key)
        if x is not None and str(x).strip() not in ('', 'None', 'nan', 'NaN')
    )
    if not text.strip():
        return False
    if _EXCLUDE_BRAND_RE.search(text):
        return True
    if _EXCLUDE_PAX_RE.search(text):
        return True
    # also catch model strings that are clearly Pax s80/s90/d210 without brand token order issues
    t = text.lower().replace('\xa0', ' ')
    if 'pax' in t and any(tok in t.replace(' ', '') for tok in ('s80', 's90', 'd210')):
        return True
    return False


amort_df['exclude_amort_by_model'] = [
    _exclude_amort_by_model(md, mk)
    for md, mk in zip(amort_df.get('model_device'), amort_df.get('model_key'))
]

_amort_before_exclude = float(
    pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0).sum()
)
_excl_mask = amort_df['exclude_amort_by_model'].astype(bool)
amort_df.loc[_excl_mask, 'amortization_monthly'] = 0.0
amort_df.loc[_excl_mask, 'amortization_for_report_month'] = 0.0
amort_df['is_amortized'] = (pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0) > 0).astype(int)

_amort_after_exclude = float(
    pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0).sum()
)
print(
    'exclude_amort_by_model rows =', int(_excl_mask.sum()),
    '| amort_sum before =', round(_amort_before_exclude, 2),
    '| after =', round(_amort_after_exclude, 2),
    '| dropped =', round(_amort_before_exclude - _amort_after_exclude, 2),
)
_excl_top = (
    amort_df.loc[_excl_mask]
    .assign(model_device=amort_df.loc[_excl_mask, 'model_device'].astype(str))
    .groupby(['model_device'], as_index=False)
    .size()
    .sort_values('size', ascending=False)
    .head(30)
)
amort_df['exclude_amort_by_model'] = amort_df['exclude_amort_by_model'].astype(int)
print('=== Top excluded models (by row count) ===')
display(_excl_top)

amort_df['load_dt'] = pd.Timestamp.now().strftime('%Y-%m-%d')
amort_df['source_file'] = source_file

qc_missing_df = (
    amort_df.groupby('qc_status', as_index=False)
    .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
    .sort_values(['rows', 'terminals_nunique'], ascending=False)
    .reset_index(drop=True)
)

join_cov_df = amort_df.copy()
join_cov_df['join_key_used'] = join_cov_df['join_key_used'].fillna('no_match')
join_coverage_df = (
    join_cov_df.groupby('join_key_used', as_index=False)
    .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
    .sort_values(['rows', 'terminals_nunique'], ascending=False)
    .reset_index(drop=True)
)

print('QC: missing reasons')
display(qc_missing_df)
print('QC: join coverage')
display(join_coverage_df)
print('amort_df rows =', len(amort_df))
print('amortized rows =', int((amort_df['is_amortized'] == 1).sum()))
display(amort_df.head(10))


## QC: модели из CDWH без цены в term_model.xlsx + складской c_nter

1. **Top-30 моделей CDWH без цены** в `term_model.xlsx`.
2. **Складской `c_nter`**: в сыром lake один номер на много serial (ожидаемо `10000051`); в `amort_df` после `rn=1` остаётся 1 serial на месяц — сравнение RAW vs post-collapse.


In [ ]:
# QC A) Top models present in CDWH mapping but missing from term_model.xlsx (no price)
has_model_no_price = amort_df.loc[
    (~amort_df['missing_model']) & amort_df['missing_price']
].copy()

top_missing_price_models = (
    has_model_no_price.groupby(['model_device', 'model_key'], as_index=False)
    .agg(
        rows=('c_nter', 'size'),
        terminals_nunique=('c_nter', 'nunique'),
        serials_nunique=('c_pos_serial', 'nunique'),
    )
    .sort_values(['terminals_nunique', 'rows'], ascending=False)
    .head(30)
    .reset_index(drop=True)
)

print('=== Top-30 models in CDWH but NOT in term_model.xlsx (missing price) ===')
if top_missing_price_models.empty:
    print('OK: no CDWH models without Excel price')
else:
    display(top_missing_price_models)

# QC B) Delivery exists, model missing (true gaps in CDWH / join)
delivery_no_model = amort_df.loc[
    (~amort_df['missing_deliver']) & amort_df['missing_model']
].copy()
top_nter_no_model = (
    delivery_no_model.groupby('c_nter', as_index=False)
    .agg(
        rows=('c_nter', 'size'),
        serials_nunique=('c_pos_serial', 'nunique'),
        sample_serial=('c_pos_serial', 'first'),
    )
    .sort_values(['serials_nunique', 'rows'], ascending=False)
    .head(30)
    .reset_index(drop=True)
)
print('=== Top-30 c_nter with delivery but NO model ===')
display(top_nter_no_model)
print(
    'rows with delivery & no model =',
    f'{len(delivery_no_model):,} / {len(amort_df):,}',
)

# QC C) Shared c_nter in CURRENT perimeter (after rn=1 collapse)
nter_serial_freq = (
    amort_df.groupby('c_nter', as_index=False)
    .agg(
        rows=('c_nter', 'size'),
        serials_nunique=('c_pos_serial', 'nunique'),
        months_nunique=('snapshot_month_start', 'nunique'),
    )
    .sort_values(['serials_nunique', 'rows'], ascending=False)
)
warehouse_like_nter = nter_serial_freq.loc[nter_serial_freq['serials_nunique'] >= 2].head(30)
print('=== Shared c_nter in amort_df (after month+c_nter rn=1 collapse), TOP-30 by serials ===')
print('NOTE: perimeter keeps only 1 serial per (month, c_nter), so warehouse id may look rare here.')
display(warehouse_like_nter)

probe_nter = '10000051'
probe_row = nter_serial_freq.loc[
    nter_serial_freq['c_nter'].map(norm_terminal_key).astype(str) == str(norm_terminal_key(probe_nter))
]
print(f'=== Frequency for c_nter={probe_nter} in amort_df (post-collapse) ===')
display(probe_row if len(probe_row) else pd.DataFrame([{'c_nter': probe_nter, 'note': 'not found in amort_df'}]))

# QC D) RAW lake: warehouse c_nter must have VERY many serials
sql_wh = (
    "select "
    "cast(c_nter as string) as c_nter, "
    "count(*) as rows_raw, "
    "count(distinct cast(c_pos_serial as string)) as serials_nunique "
    "from ods_alpha.scd1_pos_terminals "
    f"where cast(c_nter as string) = '{probe_nter}' "
    "and c_pos_serial is not null "
    "and coalesce(ods_deleted_flg, '0') <> '1' "
    "group by cast(c_nter as string)"
)
with imp:
    wh_raw_df = imp.fetch(sql_wh)
print(f'=== RAW lake serials for warehouse c_nter={probe_nter} ===')
display(
    wh_raw_df if wh_raw_df is not None and len(wh_raw_df)
    else pd.DataFrame([{'c_nter': probe_nter, 'note': 'not found in lake'}])
)

sql_top_shared = (
    "select "
    "cast(c_nter as string) as c_nter, "
    "count(distinct cast(c_pos_serial as string)) as serials_nunique, "
    "count(*) as rows_raw "
    "from ods_alpha.scd1_pos_terminals "
    "where c_nter is not null "
    "and c_pos_serial is not null "
    "and coalesce(ods_deleted_flg, '0') <> '1' "
    "group by cast(c_nter as string) "
    "having count(distinct cast(c_pos_serial as string)) >= 50 "
    "order by serials_nunique desc "
    "limit 30"
)
with imp:
    top_shared_raw = imp.fetch(sql_top_shared)
print('=== RAW lake TOP-30 shared c_nter (serials >= 50) ===')
display(top_shared_raw)


## Probe: конкретный терминал

`c_nter = 10000051`, `c_pos_serial = 2331719393`

Смотрим глазами все ступени: lake perimeter → CDWH model → Excel price → 48m window → amort amount.


In [ ]:
# Probe one terminal end-to-end
PROBE_NTER = '10000051'
PROBE_SERIAL = '2331719393'

probe_nter_norm = norm_terminal_key(PROBE_NTER)

print('=== 1) Rows in terminals_monthly_df ===')
tm = terminals_monthly_df.copy()
tm_hit = tm.loc[
    (tm['c_nter'].map(norm_terminal_key) == probe_nter_norm)
    | (tm['c_pos_serial'].astype(str).str.strip() == PROBE_SERIAL)
]
display(tm_hit.sort_values(['snapshot_month_start', 'c_pos_serial']).head(50))
print(
    'rows=', len(tm_hit),
    '| serials=', tm_hit['c_pos_serial'].nunique(),
    '| months=', tm_hit['snapshot_month_start'].nunique(),
)

print('=== 2) CDWH BM_DET_DEVICE hits for this c_nter ===')
bm_hit = bm_det_device_df.loc[
    (bm_det_device_df['device_norm'] == probe_nter_norm)
    | (bm_det_device_df['code_device_norm'] == probe_nter_norm)
    | (bm_det_device_df['device'].astype(str) == PROBE_NTER)
    | (bm_det_device_df['code_device'].astype(str) == PROBE_NTER)
]
display(bm_hit)
print(
    'in device_map =', probe_nter_norm in device_map,
    '| in code_map =', probe_nter_norm in code_map,
)
if probe_nter_norm in device_map:
    print('device_map model =', device_map[probe_nter_norm])
if probe_nter_norm in code_map:
    print('code_map model =', code_map[probe_nter_norm])

print('=== 3) amort_df rows for nter/serial ===')
ad = amort_df.copy()
ad_hit = ad.loc[
    (ad['c_nter'].map(norm_terminal_key) == probe_nter_norm)
    | (ad['c_pos_serial'].astype(str).str.strip() == PROBE_SERIAL)
]
cols_show = [
    c for c in [
        'snapshot_month_start', 'report_month', 'c_nter', 'c_pos_serial',
        'first_d_ter_delivery', 'model_device', 'model_key', 'join_key_used',
        'price', 'amortization_monthly', 'months_from_start', 'is_in_48m_window',
        'amortization_for_report_month', 'qc_status',
        'missing_serial', 'missing_deliver', 'missing_model', 'missing_price',
    ] if c in ad_hit.columns
]
display(ad_hit[cols_show].sort_values(['snapshot_month_start', 'c_pos_serial']))

print('=== 4) How many serials share this c_nter in amort_df? ===')
share = ad.loc[ad['c_nter'].map(norm_terminal_key) == probe_nter_norm]
print(
    'rows=', len(share),
    '| unique serials=', share['c_pos_serial'].nunique(),
    '| unique models=', share['model_device'].nunique(dropna=True),
)
if len(share):
    display(
        share.groupby(['model_device', 'model_key', 'qc_status'], dropna=False, as_index=False)
        .agg(rows=('c_pos_serial', 'size'), serials=('c_pos_serial', 'nunique'))
        .sort_values('serials', ascending=False)
        .head(20)
    )

print('=== 5) Exact serial probe ===')
serial_hit = ad.loc[ad['c_pos_serial'].astype(str).str.strip() == PROBE_SERIAL, cols_show]
display(serial_hit.sort_values('snapshot_month_start'))
if serial_hit.empty:
    print('WARNING: serial not found in amort_df — check lake perimeter filters / ods_deleted / date window')
else:
    print('qc_status value counts:')
    display(serial_hit['qc_status'].value_counts(dropna=False))


In [ ]:
# 5) Итоговый датафрейм для загрузки в target table
load_cols = [
    'snapshot_month_start',
    'report_month',
    'c_nter',
    'c_pos_serial',
    'first_d_ter_delivery',
    'model_device',
    'model_key',
    'price',
    'amortization_monthly',
    'months_from_start',
    'is_in_48m_window',
    'amortization_for_report_month',
    'is_amortized',
    'exclude_amort_by_model',
    'join_key_used',
    'qc_status',
    'missing_serial',
    'missing_deliver',
    'missing_model',
    'missing_price',
    'load_dt',
    'source_file',
]

for c in load_cols:
    if c not in amort_df.columns:
        amort_df[c] = None

final_load_df = amort_df[load_cols].copy()

# Однозначность ключа внутри периода
dup_check = (
    final_load_df.groupby(['snapshot_month_start', 'c_nter'], as_index=False)
    .size()
    .rename(columns={'size': 'cnt'})
)
dup_cnt = int((dup_check['cnt'] > 1).sum()) if len(dup_check) else 0
print('duplicate keys (snapshot_month_start, c_nter) =', dup_cnt)
if dup_cnt:
    display(dup_check[dup_check['cnt'] > 1].head(20))
    raise RuntimeError('Найдены дубли по ключу (snapshot_month_start, c_nter). Загрузка остановлена.')

print('final_load_df rows =', len(final_load_df))
print('final_load_df months =', sorted(final_load_df['report_month'].dropna().astype(str).unique().tolist()))
display(final_load_df.head(20))


In [ ]:
# 6) Полная перезапись target table в Datalake/Impala (DROP + CREATE + ORC load)

def load_to_datalake_from_file(local_file_path: str, table: str, dest_catalog: str = 'external', cleanup_before_copy: bool = True):
    schema = table.split('.')[0]
    table_name = table.split('.')[-1]
    hdfs_path = f'/warehouse/tablespace/{dest_catalog}/hive/{schema}.db/{table_name}'

    if cleanup_before_copy:
        # Удаляем старые файлы, чтобы избежать дублей после полной перезагрузки
        subprocess.run(['hdfs', 'dfs', '-rm', '-r', '-f', f'{hdfs_path}/*'], check=False)

    subprocess.run(['hdfs', 'dfs', '-copyFromLocal', '-f', local_file_path, f'{hdfs_path}/{local_file_path}'], check=True)
    out = subprocess.run(['hdfs', 'dfs', '-ls', '-h', hdfs_path], capture_output=True, text=True, check=True)
    print(f'Files in HDFS path {hdfs_path}:\n{out.stdout}')


# Подготовка типов и null для записи
load_df = final_load_df.copy()
for c in ['price', 'amortization_monthly', 'months_from_start', 'amortization_for_report_month']:
    load_df[c] = pd.to_numeric(load_df[c], errors='coerce')
for c in ['is_in_48m_window', 'is_amortized', 'exclude_amort_by_model', 'missing_serial', 'missing_deliver', 'missing_model', 'missing_price']:
    load_df[c] = pd.to_numeric(load_df[c], errors='coerce').fillna(0).astype('int64')

load_df = load_df.fillna({
    'snapshot_month_start': '',
    'report_month': '',
    'c_nter': '',
    'c_pos_serial': '',
    'first_d_ter_delivery': '',
    'model_device': '',
    'model_key': '',
    'join_key_used': '',
    'qc_status': '',
    'load_dt': '',
    'source_file': source_file,
})

load_df.to_orc(save_table_orc_name, index=False)
print('ORC prepared:', save_table_orc_name, 'rows=', len(load_df))

create_sql = f"""
create external table if not exists {target_table} (
    snapshot_month_start string,
    report_month string,
    c_nter string,
    c_pos_serial string,
    first_d_ter_delivery string,
    model_device string,
    model_key string,
    price double,
    amortization_monthly double,
    months_from_start double,
    is_in_48m_window bigint,
    amortization_for_report_month double,
    is_amortized bigint,
    exclude_amort_by_model bigint,
    join_key_used string,
    qc_status string,
    missing_serial bigint,
    missing_deliver bigint,
    missing_model bigint,
    missing_price bigint,
    load_dt string,
    source_file string
)
stored as orc
 tblproperties ('transactional'='false')
"""

with dl:
    dl.execute(f'drop table if exists {target_table}')
    dl.execute(create_sql)

load_to_datalake_from_file(save_table_orc_name, target_table, dest_catalog='external', cleanup_before_copy=True)

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')

print('Reload completed:', target_table)


In [ ]:
# 7) Post-load DQ checks
sql_dq = f"""
with base as (
    select *
    from {target_table}
),
dup as (
    select snapshot_month_start, c_nter, count(*) as cnt
    from base
    group by snapshot_month_start, c_nter
    having count(*) > 1
)
select
    (select count(*) from base) as rows_cnt,
    (select count(distinct c_nter) from base) as distinct_c_nter_cnt,
    (select count(distinct snapshot_month_start) from base) as months_cnt,
    (select count(*) from dup) as duplicated_month_nter_cnt,
    (select sum(case when qc_status = 'complete' then 1 else 0 end) from base) as complete_rows_cnt,
    (select sum(case when qc_status <> 'complete' then 1 else 0 end) from base) as non_complete_rows_cnt,
    (select sum(coalesce(amortization_for_report_month, 0.0)) from base) as amortization_total
"""

sql_qc_status = f"""
select
  qc_status,
  count(*) as rows_cnt,
  count(distinct c_nter) as terminals_nunique
from {target_table}
group by qc_status
order by rows_cnt desc, qc_status
"""

sql_monthly = f"""
select
  report_month,
  count(*) as rows_cnt,
  count(distinct c_nter) as terminals_nunique,
  sum(coalesce(amortization_for_report_month, 0.0)) as amortization_total
from {target_table}
group by report_month
order by report_month
"""

sql_sample = f"""
select *
from {target_table}
order by report_month, c_nter
limit 50
"""

with imp:
    dq_summary_df = imp.fetch(sql_dq)
    qc_status_df = imp.fetch(sql_qc_status)
    monthly_df = imp.fetch(sql_monthly)
    sample_df = imp.fetch(sql_sample)

print('DQ summary:')
display(dq_summary_df)
print('QC by status:')
display(qc_status_df)
print('Monthly control totals:')
display(monthly_df)
print('Sample rows:')
display(sample_df)

In [ ]:
# 7b) Smoke after reload: expect Jan–Jul including 2026-07
expected_months = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']

sql_monthly_smoke = f"""
select
  cast(report_month as string) as report_month,
  count(*) as rows_cnt,
  count(distinct cast(c_nter as string)) as terminals_nunique,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total
from {target_table}
group by cast(report_month as string)
order by 1
"""

sql_july_smoke = f"""
select
  count(*) as july_rows,
  count(distinct cast(c_nter as string)) as july_terminals,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as july_amort
from {target_table}
where cast(report_month as string) = '2026-07'
   or cast(snapshot_month_start as string) = '2026-07-01'
"""

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')
    monthly_smoke_df = imp.fetch(sql_monthly_smoke)
    july_smoke_df = imp.fetch(sql_july_smoke)

print('Smoke table:', target_table)
display(monthly_smoke_df)
display(july_smoke_df)

have = (
    set(monthly_smoke_df['report_month'].astype(str).str[:7].tolist())
    if monthly_smoke_df is not None and len(monthly_smoke_df)
    else set()
)
missing = [m for m in expected_months if m not in have]
print('months present:', sorted(have))
print('missing expected:', missing)

if missing:
    raise AssertionError(f'Missing months in amort model: {missing}')
if july_smoke_df is None or int(july_smoke_df.iloc[0]['july_rows'] or 0) <= 0:
    raise AssertionError('July 2026 is empty in amort model')

print('OK: July present, all expected months found')


## 7c) QC: объём амортизации (диагностика Excel ≪ lake)

Если lake ≫ Excel (×3–×6), проверяем:
1. долю терминалов в окне 48м (`is_in_48m_window`);
2. сумму amort / число списываемых терминалов по месяцам;
3. распределение `months_from_start`;
4. дубли ключа `(snapshot_month_start, c_nter)` в озере;
5. (опционально) один `c_nter` на нескольких `n_agr` в SA — двойной счёт в витрине.


In [ ]:
# 7c) QC: amortization volume diagnostics (post-load Impala + optional local amort_df)
from IPython.display import display

sql_vol_by_month = f"""
select
  cast(report_month as string) as report_month,
  count(*) as rows_cnt,
  sum(case when cast(is_in_48m_window as bigint) = 1 then 1 else 0 end) as in_window_cnt,
  sum(case when coalesce(cast(amortization_for_report_month as double), 0.0) > 0 then 1 else 0 end) as amort_positive_cnt,
  round(
    sum(case when cast(is_in_48m_window as bigint) = 1 then 1 else 0 end) / cast(count(*) as double),
    4
  ) as share_in_window,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total,
  avg(case
        when coalesce(cast(amortization_for_report_month as double), 0.0) > 0
        then cast(amortization_for_report_month as double)
      end) as avg_amort_when_positive,
  avg(case
        when cast(is_in_48m_window as bigint) = 1
        then cast(price as double)
      end) as avg_price_in_window
from {target_table}
group by cast(report_month as string)
order by 1
"""

sql_age_buckets = f"""
select
  cast(report_month as string) as report_month,
  case
    when months_from_start is null then 'null'
    when cast(months_from_start as double) < 0 then 'lt_0'
    when cast(months_from_start as double) < 12 then '0_11'
    when cast(months_from_start as double) < 24 then '12_23'
    when cast(months_from_start as double) < 36 then '24_35'
    when cast(months_from_start as double) < 48 then '36_47'
    else 'ge_48'
  end as age_bucket,
  count(*) as terminals_cnt,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total
from {target_table}
group by 1, 2
order by 1, 2
"""

sql_dup_keys = f"""
select snapshot_month_start, c_nter, count(*) as cnt
from {target_table}
group by snapshot_month_start, c_nter
having count(*) > 1
order by cnt desc
limit 50
"""

_sample_month = '2026-01-01'
sql_multi_agr = f"""
with amort_pos as (
  select distinct cast(c_nter as string) as c_nter
  from {target_table}
  where cast(snapshot_month_start as string) = '{_sample_month}'
    and coalesce(cast(amortization_for_report_month as double), 0.0) > 0
),
agr_map as (
  select
    cast(t.c_nter as string) as c_nter,
    cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_pos_terminals t
  join ods_alpha.scd1_agr_terms a
    on cast(a.c_nmrc as string) = cast(t.c_nmrc as string)
  join amort_pos p on p.c_nter = cast(t.c_nter as string)
  where coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and upper(trim(cast(a.cf_ter_type as string))) = 'P'
    and cast(a.d_valid_from as date) <= cast('{_sample_month}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) > cast('{_sample_month}' as date))
  group by cast(t.c_nter as string), cast(a.n_agr as string)
)
select
  count(*) as terminal_agr_pairs,
  count(distinct c_nter) as terminals,
  sum(case when agr_cnt > 1 then 1 else 0 end) as terminals_multi_agr
from (
  select c_nter, count(distinct n_agr) as agr_cnt
  from agr_map
  group by c_nter
) x
"""

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')
    vol_by_month_df = imp.fetch(sql_vol_by_month)
    age_buckets_df = imp.fetch(sql_age_buckets)
    dup_keys_df = imp.fetch(sql_dup_keys)
    try:
        multi_agr_df = imp.fetch(sql_multi_agr)
    except Exception as exc:
        print('multi-agr check skipped:', type(exc).__name__, str(exc)[:200])
        multi_agr_df = None

print('=== Amort volume by month ===')
display(vol_by_month_df)
print('=== Age buckets (months_from_start) ===')
display(age_buckets_df)
print('=== Duplicate (snapshot_month_start, c_nter) — expect 0 rows ===')
display(dup_keys_df)
if multi_agr_df is not None:
    print(f'=== Multi-agr risk for amort>0 terminals @ {_sample_month} ===')
    display(multi_agr_df)

if 'amort_df' in globals() and amort_df is not None and len(amort_df):
    _adf = amort_df.copy()
    _adf['_amort'] = pd.to_numeric(_adf.get('amortization_for_report_month'), errors='coerce').fillna(0)
    _adf['_win'] = pd.to_numeric(_adf.get('is_in_48m_window'), errors='coerce').fillna(0)
    loc = (
        _adf.assign(report_month=_adf['report_month'].astype(str))
        .groupby('report_month', as_index=False)
        .agg(
            rows=('c_nter', 'size'),
            in_window=('_win', 'sum'),
            amortization_total=('_amort', 'sum'),
        )
    )
    loc['share_in_window'] = loc['in_window'] / loc['rows']
    print('=== Local amort_df (before/alongside load) ===')
    display(loc)

if vol_by_month_df is not None and len(vol_by_month_df):
    _v = vol_by_month_df.copy()
    _v['share_in_window'] = pd.to_numeric(_v['share_in_window'], errors='coerce')
    high_share = _v.loc[_v['share_in_window'] >= 0.90, 'report_month'].astype(str).tolist()
    if high_share:
        print(
            'WARNING: share_in_window >= 90% for months:', high_share,
            '- почти весь парк в окне 48м; сверьте first_d_ter_delivery vs дату закупки Excel.'
        )
    if dup_keys_df is not None and len(dup_keys_df):
        print('WARNING: duplicate keys in amort model - витрина может раздувать SUM(amort).')
    if multi_agr_df is not None and len(multi_agr_df):
        _ma = int(pd.to_numeric(multi_agr_df.iloc[0].get('terminals_multi_agr'), errors='coerce') or 0)
        if _ma > 0:
            print(
                f'WARNING: {_ma} terminals with amort>0 map to multiple n_agr @ {_sample_month} - '
                'возможен двойной счёт в final_df при SUM по договорам.'
            )

_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')
qc_vol_csv = _qc_out_dir / 'amortization_volume_qc_by_month.csv'
if vol_by_month_df is not None:
    vol_by_month_df.to_csv(qc_vol_csv, index=False, encoding='utf-8-sig')
    print('Saved:', qc_vol_csv)


## 7d) QC: mid-month transfer serial в периметре `final_df`

Сколько физических устройств (`c_pos_serial`) в **нашем** периметре (ключи из `final_df`) в одном месяце жили у ≥2 `c_nter` / клиентов.

- Периметр: `final_df` (memory или CSV) → `agr_id`+`inn` → SA `n_agr` → `agr_terms` → `pos_terminals`.
- **handover_same_month**: ≥2 `c_nter` и ≥2 `n_cmp_client` (fallback: `n_agr`) на один serial в месяце.
- **close_then_install**: у одного `c_nter` close ∈ месяц, у другого install ∈ тот же месяц.
- Не путать с multi-agr (один `c_nter` на нескольких договорах) — это 7c.


In [ ]:
# 7d) QC: mid-month serial transfer within final_df perimeter
from IPython.display import display

FINAL_DF_CSV_CANDIDATES = [
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_07_mpos.csv'),
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_06_mpos.csv'),
]
AGR_CHUNK = 800
NAGR_CHUNK = 700
_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')


def _norm_agr_7d(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s in {'', 'nan', 'None', 'NaN'}:
        return None
    s = re.sub(r'\.0$', '', s)
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s


def _norm_inn_7d(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v).strip())
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    return s


def _load_final_df_for_7d():
    if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
        return final_df_period_df.copy(), 'memory:final_df_period_df'
    if 'final_df_by_month' in globals() and final_df_by_month:
        parts = []
        for m, g in final_df_by_month.items():
            if g is None or len(g) == 0:
                continue
            gg = g.copy()
            if 'report_month' not in gg.columns:
                gg['report_month'] = str(m)[:7]
            parts.append(gg)
        if parts:
            return pd.concat(parts, ignore_index=True), 'memory:final_df_by_month'
    for pth in FINAL_DF_CSV_CANDIDATES:
        if pth.exists():
            return pd.read_csv(pth, dtype=str, low_memory=False), f'csv:{pth}'
    raise RuntimeError(
        'final_df not found: set final_df_period_df / final_df_by_month in kernel '
        'or place final_df_period_2026_01_2026_07_mpos.csv under Equaring/Data'
    )


fdf_7d, fdf_7d_src = _load_final_df_for_7d()
print('final_df source:', fdf_7d_src, '| rows=', len(fdf_7d))

need_cols = {'report_month', 'inn', 'agr_id'}
missing_cols = need_cols - set(fdf_7d.columns)
if missing_cols:
    raise RuntimeError(f'final_df missing columns: {sorted(missing_cols)}')

keys_7d = fdf_7d[['report_month', 'inn', 'agr_id']].copy()
keys_7d['report_month'] = keys_7d['report_month'].astype(str).str.strip().str[:7]
keys_7d['inn_key'] = keys_7d['inn'].map(_norm_inn_7d)
keys_7d['agr_id_key'] = keys_7d['agr_id'].map(_norm_agr_7d)
keys_7d = keys_7d.dropna(subset=['report_month', 'inn_key', 'agr_id_key']).drop_duplicates()
print('unique (month, inn, agr_id) keys =', len(keys_7d))
print('months =', sorted(keys_7d['report_month'].unique().tolist()))

agr_ids_all = clean_keys(keys_7d['agr_id_key'].tolist())
agr_map_parts = []
with imp:
    imp.execute('set MEM_LIMIT=16g')
    for chunk in iter_chunks(agr_ids_all, AGR_CHUNK):
        agr_in = sql_in(chunk)
        sql_agr_map = f"""
        select distinct
          cast(a.abs_agr_id as string) as agr_id,
          cast(a.n_agr as string) as n_agr,
          cast(a.n_cmp_client as string) as n_cmp_client,
          cast(c.c_inn as string) as inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where cast(a.abs_agr_id as string) in ({agr_in})
          and upper(trim(cast(a.acq_class as string))) = 'SA'
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
        """
        part = imp.fetch(sql_agr_map)
        if part is not None and len(part):
            agr_map_parts.append(part)

if not agr_map_parts:
    raise RuntimeError('No SA agr_id → n_agr mapping for final_df keys')

agr_map_7d = pd.concat(agr_map_parts, ignore_index=True)
agr_map_7d['agr_id_key'] = agr_map_7d['agr_id'].map(_norm_agr_7d)
agr_map_7d['inn_key'] = agr_map_7d['inn'].map(_norm_inn_7d)
agr_map_7d['n_agr'] = agr_map_7d['n_agr'].astype(str).str.strip()
agr_map_7d['n_cmp_client'] = agr_map_7d['n_cmp_client'].astype(str).str.strip()
agr_map_7d = agr_map_7d.dropna(subset=['agr_id_key', 'inn_key', 'n_agr']).drop_duplicates(
    subset=['agr_id_key', 'inn_key', 'n_agr']
)

keys_nagr = keys_7d.merge(
    agr_map_7d[['agr_id_key', 'inn_key', 'n_agr', 'n_cmp_client']],
    on=['agr_id_key', 'inn_key'],
    how='inner',
)
print(
    'keys with n_agr =', len(keys_nagr),
    '| coverage agr keys =',
    round(
        keys_nagr[['report_month', 'agr_id_key', 'inn_key']].drop_duplicates().shape[0]
        / max(len(keys_7d), 1),
        4,
    ),
)

term_parts = []
for month_label, g_month in keys_nagr.groupby('report_month'):
    month_start = f'{month_label}-01'
    month_end = (pd.Timestamp(month_start) + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    n_agrs = clean_keys(g_month['n_agr'].tolist())
    print(f'[{month_label}] n_agr in perimeter =', len(n_agrs))
    for chunk in iter_chunks(n_agrs, NAGR_CHUNK):
        nagr_in = sql_in(chunk)
        sql_terms = f"""
        select distinct
          cast('{month_label}' as string) as report_month,
          cast('{month_start}' as string) as snapshot_month_start,
          cast(t.n_agr as string) as n_agr,
          cast(a.n_cmp_client as string) as n_cmp_client,
          cast(t.c_nmrc as string) as c_nmrc,
          cast(p.c_nter as string) as c_nter,
          cast(p.c_pos_serial as string) as c_pos_serial,
          cast(p.d_ter_install as date) as d_ter_install,
          cast(p.d_ter_close as date) as d_ter_close
        from ods_alpha.scd1_agr_terms t
        join ods_alpha.scd1_agreements a
          on cast(a.n_agr as string) = cast(t.n_agr as string)
        join ods_alpha.scd1_merchants m
          on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
        join ods_alpha.scd1_pos_terminals p
          on cast(p.c_nmrc as string) = cast(t.c_nmrc as string)
        where cast(t.n_agr as string) in ({nagr_in})
          and t.c_nmrc is not null
          and p.c_nter is not null
          and p.c_pos_serial is not null
          and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
          and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
          and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
          and cast(p.d_ter_install as date) is not null
          and cast(p.d_ter_install as date) <= cast('{month_end}' as date)
          and coalesce(cast(p.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
          and coalesce(t.ods_deleted_flg, '0') <> '1'
          and coalesce(m.ods_deleted_flg, '0') <> '1'
          and coalesce(p.ods_deleted_flg, '0') <> '1'
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and upper(trim(cast(a.acq_class as string))) = 'SA'
        """
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            part = imp.fetch(sql_terms)
        if part is not None and len(part):
            term_parts.append(part)

if not term_parts:
    raise RuntimeError('No terminals found for final_df perimeter — check agr_terms / month overlap')

term_perim_7d = pd.concat(term_parts, ignore_index=True)
for c in ['report_month', 'n_agr', 'n_cmp_client', 'c_nter', 'c_pos_serial']:
    term_perim_7d[c] = term_perim_7d[c].astype(str).str.strip()
term_perim_7d['d_ter_install'] = pd.to_datetime(term_perim_7d['d_ter_install'], errors='coerce')
term_perim_7d['d_ter_close'] = pd.to_datetime(term_perim_7d['d_ter_close'], errors='coerce')
_allowed = keys_nagr[['report_month', 'n_agr']].drop_duplicates()
term_perim_7d = term_perim_7d.merge(_allowed, on=['report_month', 'n_agr'], how='inner')
print(
    'term_perim rows =', len(term_perim_7d),
    '| serials =', term_perim_7d['c_pos_serial'].nunique(),
    '| c_nter =', term_perim_7d['c_nter'].nunique(),
)


def _client_col(df):
    cmp_n = df['n_cmp_client'].replace({'None': np.nan, 'nan': np.nan, '': np.nan})
    if cmp_n.notna().any():
        return 'n_cmp_client'
    return 'n_agr'


client_col = _client_col(term_perim_7d)
print('client grain for handover =', client_col)

serial_month = (
    term_perim_7d.groupby(['report_month', 'c_pos_serial'], as_index=False)
    .agg(
        n_c_nter=('c_nter', 'nunique'),
        n_client=(client_col, 'nunique'),
        n_agr=('n_agr', 'nunique'),
        c_nter_list=('c_nter', lambda s: '|'.join(sorted(set(s.astype(str))))),
        client_list=(client_col, lambda s: '|'.join(sorted(set(s.astype(str))))),
        n_agr_list=('n_agr', lambda s: '|'.join(sorted(set(s.astype(str))))),
    )
)


def _is_close_then_install(g, month_label):
    ms = pd.Timestamp(f'{month_label}-01')
    me = ms + pd.offsets.MonthEnd(0)
    facts = (
        g.groupby('c_nter', as_index=False)
        .agg(
            d_ter_install=('d_ter_install', 'min'),
            d_ter_close=('d_ter_close', 'max'),
        )
    )
    if len(facts) < 2:
        return False
    closed_in = facts['d_ter_close'].notna() & (facts['d_ter_close'] >= ms) & (facts['d_ter_close'] <= me)
    installed_in = facts['d_ter_install'].notna() & (facts['d_ter_install'] >= ms) & (facts['d_ter_install'] <= me)
    closers = set(facts.loc[closed_in, 'c_nter'])
    openers = set(facts.loc[installed_in, 'c_nter'])
    if not closers or not openers:
        return False
    # classic handoff: different c_nter closed vs installed in the same month
    return bool(closers - openers) or bool(openers - closers) or (len(closers) >= 2)


cti_flags = []
for (rm, serial), g in term_perim_7d.groupby(['report_month', 'c_pos_serial']):
    cti_flags.append({
        'report_month': rm,
        'c_pos_serial': serial,
        'close_then_install': _is_close_then_install(g, rm),
    })
cti_df = pd.DataFrame(cti_flags)
serial_month = serial_month.merge(cti_df, on=['report_month', 'c_pos_serial'], how='left')
serial_month['close_then_install'] = serial_month['close_then_install'].fillna(False)
serial_month['handover_same_month'] = (serial_month['n_c_nter'] >= 2) & (serial_month['n_client'] >= 2)
serial_month['extra_c_nter_rows'] = np.where(
    serial_month['handover_same_month'],
    serial_month['n_c_nter'] - 1,
    0,
)

handover_stats = (
    serial_month.groupby('report_month', as_index=False)
    .agg(
        serials_in_perimeter=('c_pos_serial', 'nunique'),
        serials_multi_c_nter=('handover_same_month', 'sum'),
        serials_close_then_install=('close_then_install', 'sum'),
        extra_c_nter_rows=('extra_c_nter_rows', 'sum'),
    )
)
handover_stats['share_handover'] = (
    handover_stats['serials_multi_c_nter'] / handover_stats['serials_in_perimeter']
).round(4)
handover_stats['share_close_then_install'] = (
    handover_stats['serials_close_then_install'] / handover_stats['serials_in_perimeter']
).round(4)

print('=== Mid-month serial handover by month (final_df perimeter) ===')
display(handover_stats)

period_row = {
    'serials_in_perimeter': int(serial_month['c_pos_serial'].nunique()),
    'serial_month_rows': int(len(serial_month)),
    'serials_multi_c_nter': int(serial_month['handover_same_month'].sum()),
    'serials_close_then_install': int(serial_month['close_then_install'].sum()),
    'extra_c_nter_rows': int(serial_month['extra_c_nter_rows'].sum()),
}
period_row['share_handover'] = round(
    period_row['serials_multi_c_nter'] / max(period_row['serial_month_rows'], 1), 4
)
print('=== Period totals (serial×month grain for shares) ===')
display(pd.DataFrame([period_row]))

sample_7d = (
    serial_month.loc[serial_month['handover_same_month']]
    .sort_values(['report_month', 'n_c_nter', 'c_pos_serial'], ascending=[True, False, True])
    .head(20)
)
print('=== Sample handover serials (up to 20) ===')
display(sample_7d[[
    'report_month', 'c_pos_serial', 'n_c_nter', 'n_client', 'n_agr',
    'c_nter_list', 'client_list', 'n_agr_list', 'close_then_install', 'extra_c_nter_rows',
]])

# multi-agr within same perimeter (one c_nter → many n_agr) for scale compare
nter_agr = (
    term_perim_7d.groupby(['report_month', 'c_nter'], as_index=False)
    .agg(n_agr=('n_agr', 'nunique'))
)
multi_agr_perim = int((nter_agr['n_agr'] >= 2).sum())
multi_agr_nters = int(nter_agr.loc[nter_agr['n_agr'] >= 2, 'c_nter'].nunique())
extra_agr_rows = int((nter_agr['n_agr'] - 1).clip(lower=0).sum())

multi_agr_7c = None
if 'multi_agr_df' in globals() and multi_agr_df is not None and len(multi_agr_df):
    try:
        multi_agr_7c = int(pd.to_numeric(multi_agr_df.iloc[0].get('terminals_multi_agr'), errors='coerce') or 0)
    except Exception:
        multi_agr_7c = None

print('=== Scale vs multi-agr ===')
print(f'handover serial×month rows: {period_row["serials_multi_c_nter"]}')
print(f'extra_c_nter_rows (est. double-count if grain=c_nter): {period_row["extra_c_nter_rows"]}')
print(
    f'multi-agr (c_nter×month with ≥2 n_agr) in same perimeter: '
    f'{multi_agr_perim} rows / {multi_agr_nters} distinct c_nter'
)
print(f'extra_agr_rows (est. double-count if SUM by agr): {extra_agr_rows}')
if multi_agr_7c is not None:
    print(f'7c multi-agr terminals (amort>0 @ sample month): {multi_agr_7c}')

share = period_row['share_handover']
extra_h = period_row['extra_c_nter_rows']
if share < 0.005 and extra_h < 100:
    verdict = (
        'РЕДКИЙ ШУМ: mid-month serial transfer почти не влияет на ×5 vs Excel; '
        'главный риск — multi-agr.'
    )
elif extra_h < max(extra_agr_rows * 0.15, 1):
    verdict = (
        f'ЗАМЕТНО МЕНЬШЕ multi-agr: extra_c_nter_rows={extra_h} << extra_agr_rows={extra_agr_rows}; '
        'handover не главный драйвер раздувания amort.'
    )
elif extra_h >= extra_agr_rows * 0.5:
    verdict = (
        f'СОПОСТАВИМО с multi-agr: extra_c_nter_rows={extra_h} vs extra_agr_rows={extra_agr_rows}; '
        'нужен dedupe по serial в месяце.'
    )
else:
    verdict = (
        f'ВТОРОСТЕПЕННЫЙ ВКЛАД: handover share={share}, extra_c_nter_rows={extra_h}; '
        f'multi-agr extra_agr_rows={extra_agr_rows} всё ещё крупнее.'
    )
print('VERDICT:', verdict)

qc_handover_csv = _qc_out_dir / 'amortization_serial_midmonth_handover_by_month.csv'
qc_sample_csv = _qc_out_dir / 'amortization_serial_midmonth_handover_sample.csv'
handover_stats.to_csv(qc_handover_csv, index=False, encoding='utf-8-sig')
sample_7d.to_csv(qc_sample_csv, index=False, encoding='utf-8-sig')
print('Saved:', qc_handover_csv)
print('Saved:', qc_sample_csv)


## 7e) QC: апрель 2026 — где расходится amort (Excel vs lake)

Цель: понять **почему** lake ≫ Excel на одном месяце (апрель), без пересборки модели.

Сравниваем три слоя:
1. **Excel** `04_Апрель_2026.xlsx` — сумма `Амортизация` / `term_cnt` по договорам.
2. **Lake model** `shestopalov_terminal_amortization_model` за `2026-04` — весь парк в таблице.
3. **Mart-эквивалент** — периметр `final_df` (апрель) + serial-dedupe owner (как section 04) → join к модели.

Разложение lake: число serial с amort>0, share in-window, avg/median price, avg amort, age buckets, TOP agr delta vs Excel.


In [ ]:
# 7e) April 2026: Excel vs lake amort decomposition
from IPython.display import display

APRIL_MONTH = '2026-04'
APRIL_SNAP = '2026-04-01'
APRIL_END = '2026-04-30'
APRIL_EXCEL = Path('/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx')
FINAL_DF_CSV_CANDIDATES = [
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_07_mpos.csv'),
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_06_mpos.csv'),
]
_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')
AGR_CHUNK = 800
NAGR_CHUNK = 700
APRIL_EXCEL_HEADER = 0  # april report uses header=0 in jan_jun config


def _norm_agr_7e(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s in {'', 'nan', 'None', 'NaN'}:
        return None
    s = re.sub(r'\.0$', '', s)
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s


def _norm_inn_7e(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v).strip())
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    return s


def _to_num(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce',
    )


def _pick_col(columns, candidates):
    cols = list(columns)
    lower = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower:
            return lower[key]
    for c in cols:
        cl = str(c).strip().lower()
        for cand in candidates:
            if str(cand).strip().lower() in cl:
                return c
    return None


# ---------- 1) Excel April ----------
if not APRIL_EXCEL.exists():
    raise RuntimeError(f'April Excel not found: {APRIL_EXCEL}')

ex_raw = pd.read_excel(APRIL_EXCEL, header=APRIL_EXCEL_HEADER)
inn_col = _pick_col(ex_raw.columns, ['ИНН', 'inn', 'INN'])
agr_col = _pick_col(ex_raw.columns, ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id', 'Договор'])
amort_col = _pick_col(
    ex_raw.columns,
    ['Амортизация', 'Аморт', 'Амортизация терминалов', 'amortization', 'Амортизация, руб'],
)
term_col = _pick_col(ex_raw.columns, ['Кол-во терминалов', 'term_cnt', 'Терминалы', 'Кол-во ТЭ'])
retl_col = _pick_col(ex_raw.columns, ['Кол-во ТТ', 'retl_cnt', 'Точки', 'Кол-во точек'])

print('Excel columns resolved:', {
    'inn': inn_col, 'agr': agr_col, 'amort': amort_col, 'term': term_col, 'retl': retl_col,
})
if agr_col is None or amort_col is None:
    raise RuntimeError('Cannot resolve agr_id / amortization columns in April Excel')

ex = pd.DataFrame({
    'agr_id_key': ex_raw[agr_col].map(_norm_agr_7e),
    'inn_key': ex_raw[inn_col].map(_norm_inn_7e) if inn_col else None,
    'amortization_excel': _to_num(ex_raw[amort_col]),
})
if term_col is not None:
    ex['term_cnt_excel'] = _to_num(ex_raw[term_col])
else:
    ex['term_cnt_excel'] = np.nan
if retl_col is not None:
    ex['retl_cnt_excel'] = _to_num(ex_raw[retl_col])
else:
    ex['retl_cnt_excel'] = np.nan

ex_agr = (
    ex.dropna(subset=['agr_id_key'])
      .groupby('agr_id_key', as_index=False)
      .agg(
          amortization_excel=('amortization_excel', 'sum'),
          term_cnt_excel=('term_cnt_excel', 'max'),
          retl_cnt_excel=('retl_cnt_excel', 'max'),
          inn_key=('inn_key', 'first'),
      )
)
excel_total = float(ex_agr['amortization_excel'].fillna(0).sum())
excel_term = float(ex_agr['term_cnt_excel'].fillna(0).sum())
excel_agr_pos = int((ex_agr['amortization_excel'].fillna(0) > 0).sum())
print('=== Excel April totals ===')
print(f'amortization_excel={excel_total:,.2f}')
print(f'term_cnt_excel_sum={excel_term:,.0f} | agr_with_amort>0={excel_agr_pos} | agr_rows={len(ex_agr)}')


# ---------- 2) Lake model April (full table) ----------
sql_lake_april = f"""
select
  count(*) as rows_cnt,
  count(distinct cast(c_nter as string)) as c_nter_nunique,
  count(distinct cast(c_pos_serial as string)) as serial_nunique,
  sum(case when cast(is_in_48m_window as bigint) = 1 then 1 else 0 end) as in_window_cnt,
  sum(case when coalesce(cast(amortization_for_report_month as double), 0.0) > 0 then 1 else 0 end) as amort_pos_cnt,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total,
  avg(case when coalesce(cast(amortization_for_report_month as double), 0.0) > 0
           then cast(amortization_for_report_month as double) end) as avg_amort_pos,
  avg(case when cast(is_in_48m_window as bigint) = 1 then cast(price as double) end) as avg_price_in_window,
  appx_median(case when cast(is_in_48m_window as bigint) = 1 then cast(price as double) end) as med_price_in_window
from {target_table}
where cast(snapshot_month_start as string) = '{APRIL_SNAP}'
   or cast(report_month as string) = '{APRIL_MONTH}'
"""
sql_lake_age = f"""
select
  case
    when months_from_start is null then 'null'
    when cast(months_from_start as double) < 0 then 'lt_0'
    when cast(months_from_start as double) < 12 then '0_11'
    when cast(months_from_start as double) < 24 then '12_23'
    when cast(months_from_start as double) < 36 then '24_35'
    when cast(months_from_start as double) < 48 then '36_47'
    else 'ge_48'
  end as age_bucket,
  count(*) as terminals_cnt,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total
from {target_table}
where cast(snapshot_month_start as string) = '{APRIL_SNAP}'
   or cast(report_month as string) = '{APRIL_MONTH}'
group by 1
order by 1
"""

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute('set MEM_LIMIT=16g')
    try:
        lake_april_full = imp.fetch(sql_lake_april)
    except Exception as exc:
        print('appx_median failed, retry without median:', type(exc).__name__)
        sql_lake_april_nomed = sql_lake_april.replace(
            "  appx_median(case when cast(is_in_48m_window as bigint) = 1 then cast(price as double) end) as med_price_in_window\n",
            "  cast(null as double) as med_price_in_window\n",
        )
        lake_april_full = imp.fetch(sql_lake_april_nomed)
    lake_april_age = imp.fetch(sql_lake_age)

print('=== Lake model April (FULL table, not only SA mart) ===')
display(lake_april_full)
display(lake_april_age)


# ---------- 3) Mart-equivalent: final_df April perimeter + serial owner + amort ----------
def _load_final_df_april():
    if 'final_df_by_month' in globals() and final_df_by_month and APRIL_MONTH in final_df_by_month:
        return final_df_by_month[APRIL_MONTH].copy(), f'memory:final_df_by_month[{APRIL_MONTH}]'
    if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
        tmp = final_df_period_df.copy()
        tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
        part = tmp.loc[tmp['report_month'] == APRIL_MONTH].copy()
        if len(part):
            return part, 'memory:final_df_period_df'
    for pth in FINAL_DF_CSV_CANDIDATES:
        if pth.exists():
            tmp = pd.read_csv(pth, dtype=str, low_memory=False)
            tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
            part = tmp.loc[tmp['report_month'] == APRIL_MONTH].copy()
            if len(part):
                return part, f'csv:{pth}'
    raise RuntimeError('April final_df not found in memory/CSV')


fdf_apr, fdf_src = _load_final_df_april()
print('final_df April source:', fdf_src, '| rows=', len(fdf_apr))

keys = fdf_apr[['inn', 'agr_id']].copy() if 'inn' in fdf_apr.columns else fdf_apr[['agr_id']].copy()
keys['agr_id_key'] = keys['agr_id'].map(_norm_agr_7e)
if 'inn' in keys.columns:
    keys['inn_key'] = keys['inn'].map(_norm_inn_7e)
else:
    keys['inn_key'] = None
keys = keys.dropna(subset=['agr_id_key']).drop_duplicates()

# lake amort on final_df rows (already mart output) if column present
if 'amortization' in fdf_apr.columns:
    fdf_amort_sum = float(pd.to_numeric(fdf_apr['amortization'], errors='coerce').fillna(0).sum())
    fdf_term_sum = float(pd.to_numeric(fdf_apr.get('term_cnt'), errors='coerce').fillna(0).sum()) if 'term_cnt' in fdf_apr.columns else np.nan
    print(f'final_df April SUM(amortization)={fdf_amort_sum:,.2f} | SUM(term_cnt)={fdf_term_sum}')
else:
    fdf_amort_sum = np.nan
    fdf_term_sum = np.nan

agr_ids = clean_keys(keys['agr_id_key'].tolist())
agr_map_parts = []
with imp:
    imp.execute('set MEM_LIMIT=16g')
    for chunk in iter_chunks(agr_ids, AGR_CHUNK):
        agr_in = sql_in(chunk)
        sql_map = f"""
        select distinct
          cast(a.abs_agr_id as string) as agr_id,
          cast(a.n_agr as string) as n_agr,
          cast(c.c_inn as string) as inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
        where cast(a.abs_agr_id as string) in ({agr_in})
          and upper(trim(cast(a.acq_class as string))) = 'SA'
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
        """
        part = imp.fetch(sql_map)
        if part is not None and len(part):
            agr_map_parts.append(part)

agr_map = pd.concat(agr_map_parts, ignore_index=True) if agr_map_parts else pd.DataFrame()
if agr_map.empty:
    raise RuntimeError('No SA agr map for April final_df keys')
agr_map['agr_id_key'] = agr_map['agr_id'].map(_norm_agr_7e)
agr_map['inn_key'] = agr_map['inn'].map(_norm_inn_7e)
agr_map['n_agr'] = agr_map['n_agr'].astype(str).str.strip()

if keys['inn_key'].notna().any():
    keys_nagr = keys.merge(agr_map[['agr_id_key', 'inn_key', 'n_agr']], on=['agr_id_key', 'inn_key'], how='inner')
else:
    keys_nagr = keys.merge(agr_map[['agr_id_key', 'n_agr']].drop_duplicates(), on='agr_id_key', how='inner')

n_agrs = clean_keys(keys_nagr['n_agr'].tolist())
print('April perimeter n_agr =', len(n_agrs))

term_parts = []
for chunk in iter_chunks(n_agrs, NAGR_CHUNK):
    nagr_in = sql_in(chunk)
    sql_terms = f"""
    with term_active as (
      select distinct
        cast(t.n_agr as string) as n_agr,
        cast(p.c_nter as string) as c_nter,
        cast(p.c_pos_serial as string) as c_pos_serial
      from ods_alpha.scd1_agr_terms t
      join ods_alpha.scd1_agreements a
        on cast(a.n_agr as string) = cast(t.n_agr as string)
      join ods_alpha.scd1_merchants m
        on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
      join ods_alpha.scd1_pos_terminals p
        on cast(p.c_nmrc as string) = cast(t.c_nmrc as string)
      where cast(t.n_agr as string) in ({nagr_in})
        and t.c_nmrc is not null
        and p.c_nter is not null
        and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
        and cast(t.d_valid_from as date) <= cast('{APRIL_END}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{APRIL_SNAP}' as date))
        and cast(p.d_ter_install as date) is not null
        and cast(p.d_ter_install as date) <= cast('{APRIL_END}' as date)
        and coalesce(cast(p.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{APRIL_SNAP}' as date)
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(m.ods_deleted_flg, '0') <> '1'
        and coalesce(p.ods_deleted_flg, '0') <> '1'
        and coalesce(a.ods_deleted_flg, '0') <> '1'
        and upper(trim(cast(a.acq_class as string))) = 'SA'
    ),
    owner as (
      select n_agr, c_nter, c_pos_serial
      from (
        select
          n_agr, c_nter, c_pos_serial,
          row_number() over (
            partition by coalesce(
              nullif(trim(c_pos_serial), ''),
              concat('__NTER__', c_nter)
            )
            order by n_agr, c_nter
          ) as rn
        from term_active
      ) z
      where rn = 1
    )
    select
      o.n_agr,
      o.c_nter,
      o.c_pos_serial,
      cast(am.price as double) as price,
      cast(am.amortization_monthly as double) as amortization_monthly,
      cast(am.amortization_for_report_month as double) as amortization_for_report_month,
      cast(am.is_in_48m_window as bigint) as is_in_48m_window,
      cast(am.months_from_start as double) as months_from_start,
      cast(am.model_device as string) as model_device,
      cast(am.first_d_ter_delivery as string) as first_d_ter_delivery
    from owner o
    left join {target_table} am
      on cast(am.c_nter as string) = o.c_nter
     and cast(am.snapshot_month_start as date) = cast('{APRIL_SNAP}' as date)
    """
    with imp:
        imp.execute('set MEM_LIMIT=16g')
        part = imp.fetch(sql_terms)
    if part is not None and len(part):
        term_parts.append(part)

mart_terms = pd.concat(term_parts, ignore_index=True) if term_parts else pd.DataFrame()
if mart_terms.empty:
    raise RuntimeError('No April mart-equivalent terminals')

for c in ['n_agr', 'c_nter', 'c_pos_serial']:
    mart_terms[c] = mart_terms[c].astype(str).str.strip()
for c in ['price', 'amortization_monthly', 'amortization_for_report_month', 'months_from_start', 'is_in_48m_window']:
    mart_terms[c] = pd.to_numeric(mart_terms[c], errors='coerce')

mart_amort_total = float(mart_terms['amortization_for_report_month'].fillna(0).sum())
mart_serials = int(mart_terms['c_pos_serial'].nunique())
mart_pos = mart_terms.loc[mart_terms['amortization_for_report_month'].fillna(0) > 0]
mart_pos_n = int(len(mart_pos))
mart_in_win = int((mart_terms['is_in_48m_window'].fillna(0) == 1).sum())
mart_avg_amort_pos = float(mart_pos['amortization_for_report_month'].mean()) if mart_pos_n else np.nan
mart_avg_price_win = float(
    mart_terms.loc[mart_terms['is_in_48m_window'].fillna(0) == 1, 'price'].mean()
)

print('=== Mart-equivalent April (final_df perimeter + serial owner + model) ===')
print(f'serials={mart_serials} | in_window={mart_in_win} | amort>0={mart_pos_n}')
print(f'amortization_mart={mart_amort_total:,.2f}')
print(f'avg_amort_when_pos={mart_avg_amort_pos:,.2f} | avg_price_in_window={mart_avg_price_win:,.2f}')
print(f'share_in_window={mart_in_win / max(mart_serials, 1):.4f} | share_amort_pos={mart_pos_n / max(mart_serials, 1):.4f}')

# map n_agr -> agr_id for Excel compare
# keys_nagr already has agr_id_key from agr_map join; avoid merge suffix _x/_y
nagr_to_agr = keys_nagr[['n_agr', 'agr_id_key']].dropna().drop_duplicates()
mart_by_agr = (
    mart_terms.merge(nagr_to_agr, on='n_agr', how='left')
    .groupby('agr_id_key', as_index=False)
    .agg(
        amortization_lake=('amortization_for_report_month', 'sum'),
        serials=('c_pos_serial', 'nunique'),
        amort_pos_serials=('amortization_for_report_month', lambda s: int((pd.to_numeric(s, errors='coerce').fillna(0) > 0).sum())),
    )
)

cmp = ex_agr.merge(mart_by_agr, on='agr_id_key', how='outer', indicator=True)
cmp['amortization_excel'] = cmp['amortization_excel'].fillna(0)
cmp['amortization_lake'] = cmp['amortization_lake'].fillna(0)
cmp['delta'] = cmp['amortization_lake'] - cmp['amortization_excel']
cmp['abs_delta'] = cmp['delta'].abs()

only_excel = int((cmp['_merge'] == 'left_only').sum())
only_lake = int((cmp['_merge'] == 'right_only').sum())
both = int((cmp['_merge'] == 'both').sum())

summary = pd.DataFrame([
    {
        'layer': 'excel_april',
        'amortization': excel_total,
        'term_or_serial': excel_term,
        'agr_with_amort_gt0': excel_agr_pos,
    },
    {
        'layer': 'lake_model_full_april',
        'amortization': float(pd.to_numeric(lake_april_full.iloc[0]['amortization_total'], errors='coerce') or 0),
        'term_or_serial': float(pd.to_numeric(lake_april_full.iloc[0]['serial_nunique'], errors='coerce') or 0),
        'agr_with_amort_gt0': float(pd.to_numeric(lake_april_full.iloc[0]['amort_pos_cnt'], errors='coerce') or 0),
    },
    {
        'layer': 'mart_equiv_serial_dedupe',
        'amortization': mart_amort_total,
        'term_or_serial': mart_serials,
        'agr_with_amort_gt0': mart_pos_n,
    },
    {
        'layer': 'final_df_april_sum',
        'amortization': fdf_amort_sum,
        'term_or_serial': fdf_term_sum,
        'agr_with_amort_gt0': np.nan,
    },
])
summary['ratio_vs_excel'] = summary['amortization'] / excel_total if excel_total else np.nan

print('=== Layer comparison (April) ===')
display(summary)
print(f'agr coverage: both={both} | only_excel={only_excel} | only_lake={only_lake}')

top_delta = cmp.sort_values('abs_delta', ascending=False).head(20)[[
    'agr_id_key', 'inn_key', 'amortization_excel', 'amortization_lake', 'delta',
    'term_cnt_excel', 'serials', 'amort_pos_serials', '_merge',
]]
print('=== TOP-20 agr by |lake - excel| amort ===')
display(top_delta)

# Decomposition / counterfactuals vs Excel
implied_excel_per_pos = excel_total / mart_pos_n if mart_pos_n else np.nan
ratio_amt = mart_amort_total / excel_total if excel_total else np.nan
ratio_avg = mart_avg_amort_pos / implied_excel_per_pos if implied_excel_per_pos else np.nan

# If Excel had same # positive serials, what avg would match Excel total?
print('=== Decomposition hints ===')
print(f'ratio mart/excel amort = {ratio_amt:.3f}')
print(f'lake amort>0 serials = {mart_pos_n}')
print(f'if Excel total spread over same {mart_pos_n} serials → avg_excel_implied = {implied_excel_per_pos:,.2f}')
print(f'lake avg_amort_pos = {mart_avg_amort_pos:,.2f} → avg ratio = {ratio_avg:.3f}')
print(
    'INTERPRET: '
    + (
        'в основном ДОРОЖЕ (цена/месяц), число списываемых близко'
        if ratio_avg > 1.5 and mart_pos_n > 0
        else 'смесь цены и числа устройств / окна'
    )
)

# How many positive serials would Excel "afford" at lake avg?
excel_implied_n_at_lake_avg = (
    excel_total / mart_avg_amort_pos if mart_avg_amort_pos and mart_avg_amort_pos > 0 else np.nan
)
print(
    f'if lake avg stayed {mart_avg_amort_pos:,.2f}, Excel total implies ~'
    f'{excel_implied_n_at_lake_avg:,.0f} serials in amort (vs lake {mart_pos_n})'
)

age_mart = (
    mart_terms.assign(
        age_bucket=pd.cut(
            mart_terms['months_from_start'],
            bins=[-np.inf, 0, 12, 24, 36, 48, np.inf],
            labels=['lt_0', '0_11', '12_23', '24_35', '36_47', 'ge_48'],
            right=False,
        ).astype(str)
    )
    .groupby('age_bucket', as_index=False)
    .agg(
        serials=('c_pos_serial', 'nunique'),
        amortization=('amortization_for_report_month', 'sum'),
        avg_price=('price', 'mean'),
    )
)
print('=== Mart age buckets (April perimeter) ===')
display(age_mart)

# price distribution of positive amort
if mart_pos_n:
    price_q = mart_pos['price'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_frame('price').reset_index()
    price_q.columns = ['quantile', 'price']
    print('=== Price quantiles among amort>0 (mart) ===')
    display(price_q)
    top_models = (
        mart_pos.groupby(mart_pos['model_device'].astype(str), as_index=False)
        .agg(
            serials=('c_pos_serial', 'nunique'),
            amortization=('amortization_for_report_month', 'sum'),
            avg_price=('price', 'mean'),
        )
        .sort_values('amortization', ascending=False)
        .head(15)
    )
    print('=== TOP-15 models by amort sum (mart, amort>0) ===')
    display(top_models)

out_summary = _qc_out_dir / 'amortization_april_2026_layer_compare.csv'
out_top = _qc_out_dir / 'amortization_april_2026_top_agr_delta.csv'
out_terms = _qc_out_dir / 'amortization_april_2026_mart_terms_sample.csv'
summary.to_csv(out_summary, index=False, encoding='utf-8-sig')
top_delta.to_csv(out_top, index=False, encoding='utf-8-sig')
mart_pos.sort_values('amortization_for_report_month', ascending=False).head(200).to_csv(
    out_terms, index=False, encoding='utf-8-sig'
)
print('Saved:', out_summary)
print('Saved:', out_top)
print('Saved:', out_terms)

# Final verdict lines
if ratio_amt > 3:
    if excel_implied_n_at_lake_avg and excel_implied_n_at_lake_avg < mart_pos_n * 0.4:
        verdict = (
            f'VERDICT: Excel списывает заметно МЕНЬШЕ устройств (или с меньшей price/48). '
            f'При lake avg={mart_avg_amort_pos:,.0f} Excel-сумме хватило бы ~{excel_implied_n_at_lake_avg:,.0f} serial '
            f'против lake {mart_pos_n}. Смотри окно 48м / дату старта / кто попадает в Excel amort.'
        )
    else:
        verdict = (
            f'VERDICT: расхождение ×{ratio_amt:.1f}. Проверь цены моделей (TOP-15) и share_in_window='
            f'{mart_in_win / max(mart_serials, 1):.1%}.'
        )
else:
    verdict = f'VERDICT: разрыв умеренный (×{ratio_amt:.2f}); смотри TOP agr delta.'
print(verdict)


## 7f) Гипотеза: Excel окно от закупки, lake — от выдачи

**Гипотеза:** у части терминалов закупка раньше `first_d_ter_delivery` → в Excel 48м уже закончились, а в lake (от выдачи) ещё идут.

Проверка на **апреле 2026**, периметр `final_df` (переиспользует объекты из **7e**, если ячейка уже бежала; иначе пересчитает минимально).

1. Найти в `scd1_pos_terminals` колонки-кандидаты на дату закупки (`describe`).
2. Если поле найдено — сравнить окна purchase vs delivery на serial.
3. Proxy без закупки: сдвинуть старт окна на N месяцев назад (3/6/12/18/24) и посмотреть, сколько serial выпадает из окна и как сумма сближается с Excel.
4. Среди agr с `excel=0` и `lake>0` — распределение `months_from_start` (delivery): много ли «хвоста» 36–47.


In [ ]:
# 7f) Hypothesis: Excel 48m from purchase vs lake from first delivery (April)
from IPython.display import display

APRIL_MONTH = '2026-04'
APRIL_SNAP = '2026-04-01'
APRIL_END = '2026-04-30'
APRIL_EXCEL = Path('/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx')
FINAL_DF_CSV_CANDIDATES = [
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_07_mpos.csv'),
    Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_06_mpos.csv'),
]
_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')
LAGS_MONTHS = [0, 3, 6, 12, 18, 24, 36]
PURCHASE_COL_CANDIDATES = [
    'd_ter_purchase', 'd_purchase', 'd_purch', 'd_buy', 'd_ter_buy',
    'd_acquisition', 'd_acq', 'd_own', 'd_ter_own', 'dt_purchase',
    'd_zakup', 'd_zakupki', 'purchase_date', 'd_ter_purch',
]


def _norm_agr_7f(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s in {'', 'nan', 'None', 'NaN'}:
        return None
    s = re.sub(r'\.0$', '', s)
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s


def _norm_inn_7f(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v).strip())
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    return s


def _to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def _pick_col(columns, candidates):
    lower = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if str(cand).strip().lower() in lower:
            return lower[str(cand).strip().lower()]
    for c in columns:
        cl = str(c).strip().lower()
        for cand in candidates:
            if str(cand).strip().lower() in cl:
                return c
    return None


def _months_between(start_ts, report_ts):
    """Full calendar months from start month to report month (same as amort model)."""
    s = pd.to_datetime(start_ts).to_period('M').to_timestamp()
    r = pd.to_datetime(report_ts).to_period('M').to_timestamp()
    return (r.year - s.year) * 12 + (r.month - s.month)


# ---------- 0) describe: purchase-like columns ----------
print('=== 0) pos_terminals columns (purchase-like) ===')
purchase_col = None
try:
    with imp:
        desc = imp.fetch('describe ods_alpha.scd1_pos_terminals')
    colname = None
    for c in desc.columns:
        if str(c).lower() in {'name', 'col_name', 'column_name', 'field'}:
            colname = c
            break
    if colname is None:
        colname = desc.columns[0]
    all_cols = [str(x).strip() for x in desc[colname].tolist()]
    print('total columns =', len(all_cols))
    hits = []
    for c in all_cols:
        cl = c.lower()
        if any(k in cl for k in ['purch', 'buy', 'zakup', 'acquis', 'own', 'deliver', 'install']):
            hits.append(c)
    print('date-ish hits:', hits)
    for cand in PURCHASE_COL_CANDIDATES:
        for c in all_cols:
            if c.lower() == cand.lower():
                purchase_col = c
                break
        if purchase_col:
            break
    # fuzzy: *purch* / *zakup*
    if purchase_col is None:
        for c in all_cols:
            cl = c.lower()
            if 'purch' in cl or 'zakup' in cl or cl.endswith('_buy') or 'd_buy' in cl:
                purchase_col = c
                break
    print('selected purchase_col =', purchase_col)
except Exception as exc:
    print('describe failed:', type(exc).__name__, str(exc)[:200])
    purchase_col = None


# ---------- 1) Ensure April mart_terms + excel agr from 7e or rebuild light ----------
need_rebuild = not (
    'mart_terms' in globals()
    and mart_terms is not None
    and len(mart_terms)
    and 'ex_agr' in globals()
    and ex_agr is not None
    and len(ex_agr)
)

if need_rebuild:
    print('7e objects missing — light rebuild for April mart_terms + Excel agr')
    if not APRIL_EXCEL.exists():
        raise RuntimeError(f'Missing Excel: {APRIL_EXCEL}')
    ex_raw = pd.read_excel(APRIL_EXCEL, header=0)
    agr_col = _pick_col(ex_raw.columns, ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'])
    amort_col = _pick_col(ex_raw.columns, ['Амортизация', 'Аморт', 'amortization', 'Амортизация, руб'])
    inn_col = _pick_col(ex_raw.columns, ['ИНН', 'inn', 'INN'])
    if agr_col is None or amort_col is None:
        raise RuntimeError('Excel agr/amort columns not found')
    ex_agr = pd.DataFrame({
        'agr_id_key': ex_raw[agr_col].map(_norm_agr_7f),
        'inn_key': ex_raw[inn_col].map(_norm_inn_7f) if inn_col else None,
        'amortization_excel': _to_num(ex_raw[amort_col]),
    })
    ex_agr = (
        ex_agr.dropna(subset=['agr_id_key'])
        .groupby('agr_id_key', as_index=False)
        .agg(amortization_excel=('amortization_excel', 'sum'), inn_key=('inn_key', 'first'))
    )

    # final_df April keys
    fdf = None
    if 'final_df_period_df' in globals() and final_df_period_df is not None:
        tmp = final_df_period_df.copy()
        tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
        fdf = tmp.loc[tmp['report_month'] == APRIL_MONTH].copy()
    if fdf is None or not len(fdf):
        for pth in FINAL_DF_CSV_CANDIDATES:
            if pth.exists():
                tmp = pd.read_csv(pth, dtype=str, low_memory=False)
                tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
                fdf = tmp.loc[tmp['report_month'] == APRIL_MONTH].copy()
                if len(fdf):
                    break
    if fdf is None or not len(fdf):
        raise RuntimeError('April final_df not found — run 7e first or provide CSV')

    keys = fdf[['inn', 'agr_id']].copy()
    keys['agr_id_key'] = keys['agr_id'].map(_norm_agr_7f)
    keys['inn_key'] = keys['inn'].map(_norm_inn_7f)
    keys = keys.dropna(subset=['agr_id_key']).drop_duplicates()
    agr_ids = clean_keys(keys['agr_id_key'].tolist())
    agr_parts = []
    with imp:
        imp.execute('set MEM_LIMIT=16g')
        for chunk in iter_chunks(agr_ids, 800):
            part = imp.fetch(f"""
            select distinct
              cast(a.abs_agr_id as string) as agr_id,
              cast(a.n_agr as string) as n_agr,
              cast(c.c_inn as string) as inn
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
            where cast(a.abs_agr_id as string) in ({sql_in(chunk)})
              and upper(trim(cast(a.acq_class as string))) = 'SA'
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
            """)
            if part is not None and len(part):
                agr_parts.append(part)
    agr_map = pd.concat(agr_parts, ignore_index=True)
    agr_map['agr_id_key'] = agr_map['agr_id'].map(_norm_agr_7f)
    agr_map['inn_key'] = agr_map['inn'].map(_norm_inn_7f)
    agr_map['n_agr'] = agr_map['n_agr'].astype(str).str.strip()
    keys_nagr = keys.merge(agr_map[['agr_id_key', 'inn_key', 'n_agr']], on=['agr_id_key', 'inn_key'], how='inner')
    n_agrs = clean_keys(keys_nagr['n_agr'].tolist())
    term_parts = []
    for chunk in iter_chunks(n_agrs, 700):
        sql_terms = f"""
        with term_active as (
          select distinct
            cast(t.n_agr as string) as n_agr,
            cast(p.c_nter as string) as c_nter,
            cast(p.c_pos_serial as string) as c_pos_serial
          from ods_alpha.scd1_agr_terms t
          join ods_alpha.scd1_agreements a on cast(a.n_agr as string) = cast(t.n_agr as string)
          join ods_alpha.scd1_merchants m on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
          join ods_alpha.scd1_pos_terminals p on cast(p.c_nmrc as string) = cast(t.c_nmrc as string)
          where cast(t.n_agr as string) in ({sql_in(chunk)})
            and p.c_nter is not null and p.c_pos_serial is not null
            and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
            and cast(t.d_valid_from as date) <= cast('{APRIL_END}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{APRIL_SNAP}' as date))
            and cast(p.d_ter_install as date) is not null
            and cast(p.d_ter_install as date) <= cast('{APRIL_END}' as date)
            and coalesce(cast(p.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{APRIL_SNAP}' as date)
            and coalesce(t.ods_deleted_flg,'0')<>'1' and coalesce(m.ods_deleted_flg,'0')<>'1'
            and coalesce(p.ods_deleted_flg,'0')<>'1' and coalesce(a.ods_deleted_flg,'0')<>'1'
            and upper(trim(cast(a.acq_class as string))) = 'SA'
        ),
        owner as (
          select n_agr, c_nter, c_pos_serial from (
            select n_agr, c_nter, c_pos_serial,
              row_number() over (
                partition by coalesce(nullif(trim(c_pos_serial),''), concat('__NTER__', c_nter))
                order by n_agr, c_nter
              ) rn
            from term_active
          ) z where rn = 1
        )
        select
          o.n_agr, o.c_nter, o.c_pos_serial,
          cast(am.price as double) as price,
          cast(am.amortization_monthly as double) as amortization_monthly,
          cast(am.amortization_for_report_month as double) as amortization_for_report_month,
          cast(am.is_in_48m_window as bigint) as is_in_48m_window,
          cast(am.months_from_start as double) as months_from_start,
          cast(am.first_d_ter_delivery as string) as first_d_ter_delivery
        from owner o
        left join {target_table} am
          on cast(am.c_nter as string) = o.c_nter
         and cast(am.snapshot_month_start as date) = cast('{APRIL_SNAP}' as date)
        """
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            part = imp.fetch(sql_terms)
        if part is not None and len(part):
            term_parts.append(part)
    mart_terms = pd.concat(term_parts, ignore_index=True)
    nagr_to_agr = keys_nagr[['n_agr', 'agr_id_key']].dropna().drop_duplicates()
else:
    print('Reusing mart_terms / ex_agr from 7e')
    if 'nagr_to_agr' not in globals() or nagr_to_agr is None or not len(nagr_to_agr):
        # rebuild mapping from keys_nagr if present
        if 'keys_nagr' in globals() and keys_nagr is not None and len(keys_nagr) and 'agr_id_key' in keys_nagr.columns:
            nagr_to_agr = keys_nagr[['n_agr', 'agr_id_key']].dropna().drop_duplicates()
        else:
            raise RuntimeError('nagr_to_agr missing — re-run 7e or set need_rebuild')

mt = mart_terms.copy()
for c in ['price', 'amortization_monthly', 'amortization_for_report_month', 'months_from_start', 'is_in_48m_window']:
    if c in mt.columns:
        mt[c] = pd.to_numeric(mt[c], errors='coerce')
mt['first_d_ter_delivery'] = pd.to_datetime(mt['first_d_ter_delivery'], errors='coerce')
mt = mt.merge(nagr_to_agr, on='n_agr', how='left')

excel_total = float(ex_agr['amortization_excel'].fillna(0).sum())
lake_total = float(mt['amortization_for_report_month'].fillna(0).sum())
print(f'Excel April amort={excel_total:,.2f} | lake mart amort={lake_total:,.2f} | ratio={lake_total/excel_total if excel_total else np.nan:.3f}')


# ---------- 2) Proxy: shift delivery start earlier by lag months ----------
report_ts = pd.Timestamp(APRIL_SNAP)
proxy_rows = []
for lag in LAGS_MONTHS:
    # start' = delivery - lag months; months_from_start' = months(start', report)
    # equivalent: months_from_start_delivery + lag
    mfs = mt['months_from_start'] + lag
    in_win = mt['first_d_ter_delivery'].notna() & mfs.notna() & (mfs >= 0) & (mfs < 48)
    # only charge if model had price/monthly (same as having amort path)
    monthly = mt['amortization_monthly'].fillna(0)
    # if amortization_monthly missing but price present
    if monthly.eq(0).all() and 'price' in mt.columns:
        monthly = mt['price'].fillna(0) / 48.0
    amort = np.where(in_win, monthly, 0.0)
    # prefer original amortization_for_report_month when lag=0 and present
    if lag == 0:
        amort = mt['amortization_for_report_month'].fillna(0).to_numpy()
        in_win = (mt['is_in_48m_window'].fillna(0) == 1) | (mt['amortization_for_report_month'].fillna(0) > 0)
    proxy_rows.append({
        'lag_months_earlier_than_delivery': lag,
        'serials_in_window': int(pd.Series(in_win).fillna(False).sum()) if lag > 0 else int((mt['is_in_48m_window'].fillna(0) == 1).sum()),
        'serials_amort_gt0': int((pd.Series(amort) > 0).sum()),
        'amortization_total': float(pd.Series(amort).sum()),
        'ratio_vs_excel': float(pd.Series(amort).sum() / excel_total) if excel_total else np.nan,
        'delta_vs_excel': float(pd.Series(amort).sum() - excel_total),
    })

proxy_df = pd.DataFrame(proxy_rows)
print('=== Proxy: window start = delivery − lag (April mart perimeter) ===')
print('If hypothesis true, larger lag → closer to Excel (~429k / ratio→1).')
display(proxy_df)

best = proxy_df.iloc[(proxy_df['amortization_total'] - excel_total).abs().argsort().values[0]]
print(
    f'Closest lag to Excel: {int(best["lag_months_earlier_than_delivery"])} months '
    f'(amort={best["amortization_total"]:,.0f}, ratio={best["ratio_vs_excel"]:.3f})'
)


# ---------- 3) Excel=0 & lake>0: age by delivery ----------
mart_by_agr = (
    mt.groupby('agr_id_key', as_index=False)
    .agg(
        amortization_lake=('amortization_for_report_month', 'sum'),
        serials=('c_pos_serial', 'nunique'),
        avg_months_from_start=('months_from_start', 'mean'),
        max_months_from_start=('months_from_start', 'max'),
        min_months_from_start=('months_from_start', 'min'),
    )
)
cmp = ex_agr.merge(mart_by_agr, on='agr_id_key', how='outer')
cmp['amortization_excel'] = cmp['amortization_excel'].fillna(0)
cmp['amortization_lake'] = cmp['amortization_lake'].fillna(0)

excel0_lake_pos = cmp.loc[(cmp['amortization_excel'] <= 0.01) & (cmp['amortization_lake'] > 0.01)].copy()
print('=== Agr where Excel≈0 but lake>0 ===')
print(
    f'agr_cnt={len(excel0_lake_pos)} | lake_amort_sum={excel0_lake_pos["amortization_lake"].sum():,.2f} '
    f'({excel0_lake_pos["amortization_lake"].sum()/lake_total:.1%} of mart total)'
)

# serial-level for those agrs
ser_e0 = mt.loc[mt['agr_id_key'].isin(excel0_lake_pos['agr_id_key'])].copy()
ser_e0_pos = ser_e0.loc[ser_e0['amortization_for_report_month'].fillna(0) > 0].copy()

def _age_bucket(x):
    if pd.isna(x):
        return 'null'
    if x < 0:
        return 'lt_0'
    if x < 12:
        return '0_11'
    if x < 24:
        return '12_23'
    if x < 36:
        return '24_35'
    if x < 48:
        return '36_47'
    return 'ge_48'

ser_e0_pos['age_bucket'] = ser_e0_pos['months_from_start'].map(_age_bucket)
age_e0 = (
    ser_e0_pos.groupby('age_bucket', as_index=False)
    .agg(
        serials=('c_pos_serial', 'nunique'),
        amortization=('amortization_for_report_month', 'sum'),
        avg_months=('months_from_start', 'mean'),
    )
    .sort_values('age_bucket')
)
print('Age buckets (delivery) for serials with lake amort>0 on Excel=0 agrs:')
display(age_e0)

# How many of these would drop if lag=12 / 24
for lag in [6, 12, 18, 24]:
    mfs2 = ser_e0_pos['months_from_start'] + lag
    still = (mfs2 >= 0) & (mfs2 < 48)
    drop_n = int((~still).sum())
    drop_amt = float(ser_e0_pos.loc[~still, 'amortization_for_report_month'].sum())
    print(
        f'  if start = delivery-{lag}m: drop {drop_n} serials / {drop_amt:,.0f} amort '
        f'from Excel=0&lake>0 set ({drop_n/max(len(ser_e0_pos),1):.1%})'
    )


# ---------- 4) Real purchase date if column exists ----------
purchase_cmp = None
if purchase_col:
    print(f'=== 4) Real purchase window using column `{purchase_col}` ===')
    serials = clean_keys(mt['c_pos_serial'].tolist())
    purch_parts = []
    for chunk in iter_chunks(serials, 800):
        sql_p = f"""
        select
          cast(t.c_pos_serial as string) as c_pos_serial,
          min(cast(t.`{purchase_col}` as date)) as first_purchase_date,
          min(cast(t.d_ter_delivery as date)) as first_delivery_date
        from ods_alpha.scd1_pos_terminals t
        where cast(t.c_pos_serial as string) in ({sql_in(chunk)})
          and coalesce(t.ods_deleted_flg, '0') <> '1'
        group by cast(t.c_pos_serial as string)
        """
        with imp:
            part = imp.fetch(sql_p)
        if part is not None and len(part):
            purch_parts.append(part)
    if purch_parts:
        purch = pd.concat(purch_parts, ignore_index=True)
        purch['c_pos_serial'] = purch['c_pos_serial'].astype(str).str.strip()
        purch['first_purchase_date'] = pd.to_datetime(purch['first_purchase_date'], errors='coerce')
        purch['first_delivery_date'] = pd.to_datetime(purch['first_delivery_date'], errors='coerce')
        purch['purchase_before_delivery_days'] = (
            purch['first_delivery_date'] - purch['first_purchase_date']
        ).dt.days
        m2 = mt.merge(purch, on='c_pos_serial', how='left')
        m2['mfs_purchase'] = m2['first_purchase_date'].map(
            lambda d: _months_between(d, report_ts) if pd.notna(d) else np.nan
        )
        m2['in_win_purchase'] = m2['mfs_purchase'].notna() & (m2['mfs_purchase'] >= 0) & (m2['mfs_purchase'] < 48)
        monthly = m2['amortization_monthly'].fillna(0)
        if monthly.eq(0).all():
            monthly = m2['price'].fillna(0) / 48.0
        m2['amort_purchase'] = np.where(m2['in_win_purchase'], monthly, 0.0)
        m2['in_win_delivery'] = m2['is_in_48m_window'].fillna(0) == 1
        m2['only_delivery_in_window'] = m2['in_win_delivery'] & (~m2['in_win_purchase'].fillna(False))

        purchase_cmp = pd.DataFrame([{
            'serials': int(m2['c_pos_serial'].nunique()),
            'has_purchase_date': int(m2['first_purchase_date'].notna().sum()),
            'purchase_before_delivery_cnt': int((m2['purchase_before_delivery_days'] > 0).sum()),
            'avg_purchase_before_delivery_days': float(m2.loc[m2['purchase_before_delivery_days'] > 0, 'purchase_before_delivery_days'].mean())
                if (m2['purchase_before_delivery_days'] > 0).any() else np.nan,
            'amort_delivery_window': float(m2['amortization_for_report_month'].fillna(0).sum()),
            'amort_purchase_window': float(m2['amort_purchase'].sum()),
            'serials_only_in_delivery_window': int(m2['only_delivery_in_window'].sum()),
            'amort_only_in_delivery_window': float(
                m2.loc[m2['only_delivery_in_window'], 'amortization_for_report_month'].fillna(0).sum()
            ),
            'ratio_purchase_vs_excel': float(m2['amort_purchase'].sum() / excel_total) if excel_total else np.nan,
        }])
        print('Purchase vs delivery window:')
        display(purchase_cmp)
        print(
            'HYPOTHESIS SUPPORT: '
            f'{int(purchase_cmp.iloc[0]["serials_only_in_delivery_window"])} serials still in delivery-window '
            f'but OUT of purchase-window; their lake amort = '
            f'{purchase_cmp.iloc[0]["amort_only_in_delivery_window"]:,.0f}'
        )
else:
    print('=== 4) No purchase date column found — rely on proxy lags + Excel=0 age profile ===')


# ---------- verdict ----------
# Support score from proxy: does lag reduce toward excel?
ratio0 = float(proxy_df.loc[proxy_df['lag_months_earlier_than_delivery'] == 0, 'ratio_vs_excel'].iloc[0])
ratio12 = float(proxy_df.loc[proxy_df['lag_months_earlier_than_delivery'] == 12, 'ratio_vs_excel'].iloc[0])
ratio24 = float(proxy_df.loc[proxy_df['lag_months_earlier_than_delivery'] == 24, 'ratio_vs_excel'].iloc[0])
e0_share = float(excel0_lake_pos['amortization_lake'].sum() / lake_total) if lake_total else np.nan
tail_3647 = 0.0
if len(age_e0):
    row = age_e0.loc[age_e0['age_bucket'] == '36_47']
    if len(row):
        tail_3647 = float(row.iloc[0]['amortization'] / max(excel0_lake_pos['amortization_lake'].sum(), 1))

print('=== VERDICT ===')
print(f'Excel=0 & lake>0 share of mart amort: {e0_share:.1%}')
print(f'Proxy ratios vs Excel: lag0={ratio0:.2f}, lag12={ratio12:.2f}, lag24={ratio24:.2f}')
if purchase_cmp is not None:
    print(
        f'With real purchase col: purchase-window ratio vs Excel = '
        f'{purchase_cmp.iloc[0]["ratio_purchase_vs_excel"]:.2f}; '
        f'only-delivery-window amort = {purchase_cmp.iloc[0]["amort_only_in_delivery_window"]:,.0f}'
    )
if ratio24 < ratio0 * 0.7 or (purchase_cmp is not None and purchase_cmp.iloc[0]['amort_only_in_delivery_window'] > 0.2 * lake_total):
    print(
        'SUPPORT: сдвиг/закупка заметно режет окно → гипотеза правдоподобна '
        '(часть ×4.2 объясняется более ранним стартом в Excel).'
    )
elif ratio12 < ratio0 * 0.85:
    print('WEAK SUPPORT: умеренный эффект от сдвига на 12м; гипотеза часть объяснения, не всё.')
else:
    print(
        'WEAK/NO SUPPORT from lag proxy: даже delivery−24m слабо сближает с Excel → '
        'кроме даты старта есть другой фильтр Excel (кто вообще в amort).'
    )

proxy_df.to_csv(_qc_out_dir / 'amortization_april_2026_purchase_vs_delivery_proxy.csv', index=False, encoding='utf-8-sig')
age_e0.to_csv(_qc_out_dir / 'amortization_april_2026_excel0_lakepos_age.csv', index=False, encoding='utf-8-sig')
excel0_lake_pos.sort_values('amortization_lake', ascending=False).head(50).to_csv(
    _qc_out_dir / 'amortization_april_2026_excel0_lakepos_top_agr.csv', index=False, encoding='utf-8-sig'
)
print('Saved proxy / excel0 age / top agr CSVs under Equaring/Data')


## 7g) Справочник коллеги: модели с `Амортизация = 0`

В файле 503/моделей у части строк **Стоимость > 0**, но **Амортизация = 0** (Ingenico ICT/IWL, PAX S80/D210, Verifone Vx520/Vx680…).  
Lake сейчас списывает `price/48` для любой модели с ценой в `term_model.xlsx` → возможный главный источник ×4 vs Excel.

Проверка (апрель, периметр mart из **7e**):
1. Загрузить справочник коллеги (если файл есть) **или** fallback-список моделей с фото.
2. Пометить `ref_amort_zero` / `ref_amort_active`.
3. Сколько lake-amort приходится на модели с ref amort=0.
4. TOP моделей lake, попавших в «запрет» справочника.


In [ ]:
# 7g) Colleague model ref: Amortization=0 models vs lake April mart
from IPython.display import display

_qc_out_dir = Path('/home/jovyan/documents/Equaring/Data')
COLLEAGUE_REF_CANDIDATES = [
    _qc_out_dir / 'term_model_503_amort.xlsx',
    _qc_out_dir / 'term_model_amort_ref.xlsx',
    _qc_out_dir / '503_модели_амортизация.xlsx',
    _qc_out_dir / 'модели_амортизация.xlsx',
    _qc_out_dir / 'term_model.xlsx',  # if colleague added Amortization col here
]

# From colleague screenshot: Стоимость > 0, Амортизация = 0
FALLBACK_ZERO_AMORT_MODELS = [
    'ict250 e+g+cl',
    'INGENICO ICT220 e+g',
    'INGENICO ICT220 ethernet',
    'INGENICO IPP320',
    'INGENICO IWL220',
    'INGENICO IWL255cl',
    'ipp350cl',
    'IRAS 900 K',
    'PAX D210cl',
    'PAX D210wifi+cl',
    'PAX S80 e+3g+cl',
    'PAX S80 e+cl',
    'PAX S80 e+g+cl',
    'PAX S90 g+cl',
    'VERIFONE Vx520 e+g+b+cl',
    'VERIFONE Vx520 e+g+cl',
    'VERIFONE Vx520 ethernet',
    'VERIFONE Vx675',
    'VERIFONE Vx675 cl',
    'VERIFONE vx680 g+cl',
    'VERIFONE vx680 wifi+cl',
    'VERIFONE Vx805 cl',
    'VERIFONE Vx820 cl',
    'Vx520 e+cl',
]

# Also cost=0 & amort=0 (never charge)
FALLBACK_BOTH_ZERO_MODELS = [
    'AF6 4gbtwifi',
    'aQsi-5Ф',
    'INGENICO 5100',
    'INGENICO IWL250cl',
    'PAX IM20',
    'PAX S920 btwifi',
    'PAX S920 3gbtwifi',
    'SOFTPOS',
    'WIZARPOS Q2',
    'WIZARPOS Q2 MTC',
]


def _pick_col(columns, candidates):
    lower = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if str(cand).strip().lower() in lower:
            return lower[str(cand).strip().lower()]
    for c in columns:
        cl = str(c).strip().lower()
        for cand in candidates:
            if str(cand).strip().lower() in cl:
                return c
    return None


def _to_num(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce',
    )


# ---------- 1) Load colleague ref ----------
ref_src = None
ref_df = None
for pth in COLLEAGUE_REF_CANDIDATES:
    if not pth.exists():
        continue
    try:
        raw = pd.read_excel(pth)
    except Exception as exc:
        print(f'skip {pth.name}: {type(exc).__name__}')
        continue
    model_col = _pick_col(
        raw.columns,
        ['Модель по 503 отчету', 'Модель', 'model_name', 'model', 'model_device', 'Модель терминала'],
    )
    price_col = _pick_col(raw.columns, ['Стоимость', 'price', 'Цена', 'cost'])
    amort_col = _pick_col(raw.columns, ['Амортизация', 'amortization', 'Аморт', 'amort'])
    if model_col is None:
        print(f'{pth.name}: no model col, columns={list(raw.columns)}')
        continue
    if amort_col is None and pth.name == 'term_model.xlsx':
        print(f'{pth.name}: no Амортизация column — keep searching / use fallback list')
        continue
    if amort_col is None:
        print(f'{pth.name}: has model but no amort col={list(raw.columns)}')
        continue
    ref_df = pd.DataFrame({
        'model_name': raw[model_col].astype(str),
        'price_ref': _to_num(raw[price_col]) if price_col else np.nan,
        'amort_ref': _to_num(raw[amort_col]),
    })
    ref_src = f'file:{pth}'
    break

if ref_df is None:
    print('Colleague Excel with Амортизация not found in Equaring/Data — using FALLBACK list from screenshot')
    zero_names = FALLBACK_ZERO_AMORT_MODELS + FALLBACK_BOTH_ZERO_MODELS
    ref_df = pd.DataFrame({
        'model_name': zero_names,
        'price_ref': np.nan,
        'amort_ref': 0.0,
    })
    ref_src = 'fallback_screenshot_zero_amort_list'

ref_df['model_key'] = ref_df['model_name'].apply(norm_model)
ref_df = ref_df.dropna(subset=['model_key']).drop_duplicates(subset=['model_key'], keep='first')
ref_df['ref_amort_zero'] = ref_df['amort_ref'].fillna(0).abs() < 0.01
ref_df['ref_has_price'] = ref_df['price_ref'].fillna(0) > 0
ref_df['ref_flag'] = np.where(
    ref_df['ref_amort_zero'] & ref_df['ref_has_price'],
    'price_gt0_amort_0',
    np.where(ref_df['ref_amort_zero'], 'both_zero_or_no_price', 'amort_active'),
)

print('ref source:', ref_src)
print('ref models:', len(ref_df),
      '| amort_zero:', int(ref_df['ref_amort_zero'].sum()),
      '| price>0 & amort=0:', int((ref_df['ref_flag'] == 'price_gt0_amort_0').sum()),
      '| amort_active:', int((ref_df['ref_flag'] == 'amort_active').sum()))
display(ref_df['ref_flag'].value_counts(dropna=False).rename_axis('ref_flag').reset_index(name='models'))
display(ref_df.loc[ref_df['ref_amort_zero'], ['model_name', 'model_key', 'price_ref', 'amort_ref', 'ref_flag']].head(40))


# ---------- 2) Need mart_terms from 7e ----------
if 'mart_terms' not in globals() or mart_terms is None or not len(mart_terms):
    raise RuntimeError('mart_terms missing — сначала запусти 7e (апрельский mart-эквивалент)')

mt = mart_terms.copy()
for c in ['price', 'amortization_for_report_month', 'amortization_monthly']:
    if c in mt.columns:
        mt[c] = pd.to_numeric(mt[c], errors='coerce')
mt['model_device'] = mt.get('model_device', pd.Series(index=mt.index, dtype=object)).map(clean_cdwh_text) if 'clean_cdwh_text' in globals() else mt.get('model_device')
mt['model_key'] = mt['model_device'].apply(norm_model)

ref_keys_zero = set(ref_df.loc[ref_df['ref_amort_zero'], 'model_key'].tolist())
ref_keys_active = set(ref_df.loc[~ref_df['ref_amort_zero'], 'model_key'].tolist())

mt['in_ref'] = mt['model_key'].isin(set(ref_df['model_key']))
mt['ref_amort_zero'] = mt['model_key'].isin(ref_keys_zero)
mt['ref_amort_active'] = mt['model_key'].isin(ref_keys_active)
mt['amort'] = mt['amortization_for_report_month'].fillna(0)

lake_total = float(mt['amort'].sum())
zero_mask = mt['ref_amort_zero'] & (mt['amort'] > 0)
active_mask = mt['ref_amort_active'] & (mt['amort'] > 0)
unknown_mask = (~mt['in_ref']) & (mt['amort'] > 0)

summary = pd.DataFrame([
    {
        'bucket': 'ref_amort_zero (should be 0 in Excel logic)',
        'serials_amort_gt0': int(mt.loc[zero_mask, 'c_pos_serial'].nunique()) if 'c_pos_serial' in mt.columns else int(zero_mask.sum()),
        'amortization': float(mt.loc[zero_mask, 'amort'].sum()),
        'share_of_mart': float(mt.loc[zero_mask, 'amort'].sum() / lake_total) if lake_total else np.nan,
    },
    {
        'bucket': 'ref_amort_active',
        'serials_amort_gt0': int(mt.loc[active_mask, 'c_pos_serial'].nunique()) if 'c_pos_serial' in mt.columns else int(active_mask.sum()),
        'amortization': float(mt.loc[active_mask, 'amort'].sum()),
        'share_of_mart': float(mt.loc[active_mask, 'amort'].sum() / lake_total) if lake_total else np.nan,
    },
    {
        'bucket': 'model not in colleague ref',
        'serials_amort_gt0': int(mt.loc[unknown_mask, 'c_pos_serial'].nunique()) if 'c_pos_serial' in mt.columns else int(unknown_mask.sum()),
        'amortization': float(mt.loc[unknown_mask, 'amort'].sum()),
        'share_of_mart': float(mt.loc[unknown_mask, 'amort'].sum() / lake_total) if lake_total else np.nan,
    },
    {
        'bucket': 'ALL mart',
        'serials_amort_gt0': int((mt['amort'] > 0).sum()),
        'amortization': lake_total,
        'share_of_mart': 1.0,
    },
])

excel_april = None
if 'ex_agr' in globals() and ex_agr is not None and len(ex_agr):
    excel_april = float(pd.to_numeric(ex_agr['amortization_excel'], errors='coerce').fillna(0).sum())
elif 'excel_total' in globals():
    excel_april = float(excel_total)

amort_if_zero_models_dropped = lake_total - float(mt.loc[zero_mask, 'amort'].sum())
print('=== Lake April mart amort by colleague-ref flag ===')
display(summary)
if excel_april is not None:
    print(f'Excel April = {excel_april:,.2f}')
    print(f'Lake mart = {lake_total:,.2f} (ratio {lake_total/excel_april:.3f})')
    print(
        f'If drop ref_amort_zero models: lake→ {amort_if_zero_models_dropped:,.2f} '
        f'(ratio {amort_if_zero_models_dropped/excel_april:.3f})'
    )

# TOP models in forbidden bucket
top_zero = (
    mt.loc[zero_mask]
    .assign(model_device=mt.loc[zero_mask, 'model_device'].astype(str))
    .groupby(['model_key', 'model_device'], as_index=False)
    .agg(
        serials=('c_pos_serial', 'nunique') if 'c_pos_serial' in mt.columns else ('c_nter', 'nunique'),
        amortization=('amort', 'sum'),
        avg_price=('price', 'mean'),
    )
    .sort_values('amortization', ascending=False)
    .head(20)
)
print('=== TOP-20 lake models that colleague ref marks amort=0 ===')
display(top_zero)

top_unknown = (
    mt.loc[unknown_mask]
    .assign(model_device=mt.loc[unknown_mask, 'model_device'].astype(str))
    .groupby(['model_key', 'model_device'], as_index=False)
    .agg(
        serials=('c_pos_serial', 'nunique') if 'c_pos_serial' in mt.columns else ('c_nter', 'nunique'),
        amortization=('amort', 'sum'),
        avg_price=('price', 'mean'),
    )
    .sort_values('amortization', ascending=False)
    .head(15)
)
print('=== TOP-15 lake models NOT in colleague ref (name mismatch risk) ===')
display(top_unknown)

# Match quality: how many lake model_keys hit ref
lake_keys = set(mt.loc[mt['amort'] > 0, 'model_key'].dropna().tolist())
print(
    'model_key coverage among amort>0:',
    f'in_ref={len(lake_keys & set(ref_df["model_key"]))}',
    f'zero_ref={len(lake_keys & ref_keys_zero)}',
    f'active_ref={len(lake_keys & ref_keys_active)}',
    f'unknown={len(lake_keys - set(ref_df["model_key"]))}',
    f'total_pos_keys={len(lake_keys)}',
)

zero_share = float(mt.loc[zero_mask, 'amort'].sum() / lake_total) if lake_total else 0.0
print('=== VERDICT ===')
if ref_src.startswith('fallback'):
    print('NOTE: used screenshot fallback list — положи файл коллеги в Equaring/Data для полного покрытия имён.')
if zero_share >= 0.5:
    print(
        f'STRONG SUPPORT: {zero_share:.1%} mart-amort сидит на моделях с Амортизация=0 в справочнике. '
        'В lake нужно обнулять amort для этих model_key (даже при price>0).'
    )
elif zero_share >= 0.2:
    print(
        f'PARTIAL SUPPORT: {zero_share:.1%} mart-amort на ref amort=0. '
        'Часть разрыва закрывается; смотри unknown/name mismatch.'
    )
else:
    print(
        f'WEAK by current ref match ({zero_share:.1%}). '
        'Либо имена моделей не матчятся (см. TOP unknown), либо нужен полный файл коллеги.'
    )

summary.to_csv(_qc_out_dir / 'amortization_april_2026_model_ref_zero_summary.csv', index=False, encoding='utf-8-sig')
top_zero.to_csv(_qc_out_dir / 'amortization_april_2026_model_ref_zero_top.csv', index=False, encoding='utf-8-sig')
ref_df.to_csv(_qc_out_dir / 'amortization_model_ref_used.csv', index=False, encoding='utf-8-sig')
print('Saved CSVs under Equaring/Data')
